# Pipeline Experimental para Segmentação Médica Tridimensional com MONAI e PyTorch

### Segmentação Hepática 3D · Medical Segmentation Decathlon — Task03_Liver

**Versão 3.1 — Perda ciente de desbalanceamento**

**Ambiente oficial:** Google Colab · Python 3.12 · PyTorch 2.x · MONAI 1.6

---

> ⚠️ **Aviso**
>
> Este notebook implementa a visão científica, arquitetural e metodológica descrita no
> *Documento de Projeto Científico (DPC)*. O pipeline destina-se **exclusivamente a
> pesquisa, ensino e desenvolvimento tecnológico**. Embora utilize dados médicos reais e
> metodologias consolidadas na literatura, **não constitui um dispositivo médico** nem deve
> ser usado para decisão clínica sem validação específica, aprovação ética quando aplicável
> e conformidade regulatória vigente.

---

#### Sobre este notebook

Este é um **notebook monolítico organizado em camadas**: todo o fluxo experimental — do
ambiente à análise dos resultados — vive em um único arquivo, documentado em ordem
cronológica. A escolha é deliberada (DPC §5.2): facilita a **revisão metodológica** e a
compreensão por pesquisadores com pouca experiência em engenharia de software, sem abrir mão
da **modularidade lógica** entre as seções.

O texto foi escrito para ser lido tanto por profissionais de tecnologia quanto por
profissionais de saúde. Sempre que uma decisão metodológica for tomada, ela será
**justificada explicitamente** — nada relevante fica implícito no código (princípio da
*Transparência*, DPC §5.1).

> 🔬 **O que muda da v3.0 para a v3.1.** Um eixo, e apenas um: a **função de perda**. A v3.0
> mostrou que o pipeline multiclasse funciona para o fígado (Dice ≈ 0,83) e **falha por completo
> no tumor** (Dice = 0,000). Duas causas concorrem para esse zero: **(i)** a `DiceCELoss` é
> **cega ao desbalanceamento** — o tumor ocupa uma fração ínfima do volume e quase não pesa no
> gradiente; e **(ii)** o **redimensionamento global** para `(128, 128, 64)` encolhe as lesões.
> A v3.1 ataca **somente a causa (i)**.
>
> 🧩 **Nascida parametrizada, de novo.** Assim como a v3.0 introduziu o objeto `Problema`, a v3.1
> introduz a **fábrica de perdas** `criar_perda()` e o seletor **`PERDA`** (Seção 7.1), espelhando
> `criar_modelo()` + `ARQUITETURA` da Seção 6. O braço de **controle** (`dice_ce` — exatamente a
> perda da v3.0) **permanece na fábrica**: assim o controle roda **nesta mesma sessão**, com a
> mesma `SEED` e o mesmo cache, em vez de ser confrontado com números obtidos noutra execução.
>
> 🚧 **Todo o resto permanece constante.** Pré-processamento, `CONFIG`, divisão dos dados,
> métrica, critério de seleção do melhor modelo e a fábrica de arquiteturas: **intocados**. A
> amostragem focada na lesão — que ataca a causa **(ii)** — é o eixo da **v3.2**. Um eixo por vez
> (DPC §5.1, *Evolução incremental*).
>
> 🎯 **Hipótese registrada ANTES do resultado (§8.2).** **H5** — *uma perda ciente de
> desbalanceamento tira o Dice de tumor de zero, sem degradar o fígado.* Se a v3.1 devolver tumor
> ainda ≈ 0, isso **não é fracasso**: é a evidência que descarta a causa (i) como explicação
> suficiente e converte a v3.2 numa hipótese **testada**, não num palpite.
>
> ⚠️ **Ilha de comparabilidade.** A tarefa não mudou — o `Problema` é o mesmo da v3.0 —, então os
> números continuam **comparáveis aos da v3.0** e **não** aos do fígado binário (v1.0–v2.0). O que
> mudou é um elemento do *protocolo*, e é por isso que a perda passa a ser registrada em cada
> resultado (Seção 10.3) e a compor a chave da pasta (Seção 7.3).
>
> ▶️ **Como executar.** Mantenha `ARQUITETURA = "unet"` (a **âncora** desta versão, Seção 6),
> escolha a `PERDA` (Seção 7.1) e rode de ponta a ponta — **uma vez por perda**. Os resultados vão
> para `resultados/figado_multiclasse/<arquitetura>__<perda>/`; a **Seção 11** agrega e compara.


## 🗺️ Roadmap do notebook

O pipeline é uma sequência de camadas independentes (DPC §5.5), cada uma recebendo uma
entrada e produzindo a saída da etapa seguinte, **sem dependências circulares**. No notebook,
essas camadas aparecem como seções sequenciais:

| # | Seção | Status |
|---|-------|--------|
| **1** | **Configuração do Ambiente** — dependências, GPU/CPU, proveniência | ✅ **implementada** |
| **2** | **Configuração Global** — sementes, hiperparâmetros e **objeto `Problema`** | ✅ **implementada** |
| **3** | **Preparação e Exploração do Dataset** | ✅ **implementada** |
| **4** | **Pré-processamento (transforms MONAI)** | ✅ **implementada** |
| **5** | **DataLoaders** | ✅ **implementada** |
| **6** | **Definição da Arquitetura (fábrica: U-Net · SegResNet · DynUNet · UNETR · SwinUNETR)** | ✅ **implementada** |
| **7** | **Treinamento** — inclui a **fábrica de perdas** (`PERDA`) | ✅ **implementada** |
| **8** | **Inferência (*sliding window*)** | ✅ **implementada** |
| **9** | **Avaliação Quantitativa e Qualitativa (por classe)** | ✅ **implementada** |
| **10** | **Visualização e Persistência** | ✅ **implementada** |
| **11** | **Comparação entre Experimentos (arquitetura × perda, por classe)** | ✅ **implementada** |
| **12** | **Conclusões** | ✅ **implementada** |

> 💡 Em relação à v3.0, a v3.1 introduz a **fábrica de perdas** `criar_perda()` e o seletor
> **`PERDA`** (Seção 7.1), passa a versionar os resultados por **`<arquitetura>__<perda>`** com
> **guarda anti-sobrescrita** (Seção 7.3) e estende a Seção 11 para comparar **perdas**, não só
> arquiteturas. O objeto `Problema`, o pré-processamento e o restante do **protocolo** permanecem
> idênticos — a perda é o **único** eixo modificado.

# 1 · Configuração do Ambiente

A **primeira camada** do pipeline (DPC §5.5) tem uma responsabilidade simples de enunciar,
porém decisiva para a ciência do projeto: **deixar o ambiente de execução pronto e
documentado**. Nada de dados ou modelos ainda — apenas a fundação sobre a qual todo o resto
será construído.

**Por que isso vem antes de tudo?** Porque a reprodutibilidade (DPC §6.6) começa aqui. Um
resultado científico só tem valor se puder ser **reconstruído de forma independente** — e
isso exige saber, com precisão, *em que ambiente* ele foi produzido: qual versão de cada
biblioteca, qual dispositivo de hardware, em que data.

Nesta seção nós vamos:

1. **Detectar o ambiente** de execução (Google Colab ou máquina local);
2. **Instalar as dependências** com versões fixadas, reutilizando o PyTorch já presente no Colab;
3. **Configurar o dispositivo** de computação (GPU se disponível, com fallback para CPU);
4. **Registrar a proveniência** — um "cartão de identidade" do ambiente, para logging dos experimentos.

> 📌 **Fora do escopo desta seção.** As **sementes aleatórias** (`set_determinism`) e os
> **hiperparâmetros** pertencem conceitualmente à camada de *Configuração Global* e serão
> definidos na **Seção 2**. Mantemos aqui apenas o que diz respeito ao *ambiente*, para que
> cada seção tenha uma responsabilidade única e clara (DPC §5.1, Modularidade).

## 1.1 · Detecção do ambiente

O **ambiente oficial do projeto é o Google Colab** (DPC §5.3), que reduz barreiras de
infraestrutura e dá acesso relativamente simples a GPUs. Ainda assim, escrevemos o notebook
para funcionar também localmente. A célula abaixo apenas descobre **onde** estamos rodando —
essa informação orienta a instalação de dependências no passo seguinte.

In [ ]:
# Detecta se o notebook está sendo executado no Google Colab.
# A biblioteca `google.colab` só existe dentro do Colab; sua ausência indica ambiente local.
try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

print(f"Executando no Google Colab? {'sim' if EM_COLAB else 'não (ambiente local)'}")

## 1.2 · Instalação de dependências

Seguimos o princípio do DPC §5.4: **em vez de reimplementar componentes fundamentais, usamos
o MONAI**, que já oferece transformações médicas, arquiteturas 3D, inferência por janelas
deslizantes, métricas e funções de perda — tudo validado pela comunidade.

**Estratégia de versões (DPC §6.6 — "fixar versões sempre que possível"):**

- **PyTorch:** *não* é reinstalado. O Colab já traz uma build de PyTorch casada com a versão
  de CUDA do ambiente; reinstalá-lo é a principal fonte de incompatibilidades. Reutilizamos o
  que está presente e apenas **verificamos** a versão no passo 1.4.
- **MONAI:** fixado em `monai==1.6.0` — **exatamente a versão que o Colab já entrega
  pré-instalada** no momento desta escrita. Por que fixar em algo já presente? Porque o pin
  (a) *documenta* qual versão foi testada e (b) protege o experimento caso, no futuro, o Colab
  passe a trazer uma versão diferente. Como a versão fixada coincide com a pré-instalada, o
  comando é **idempotente** (o pip responde "already satisfied" e não altera nada) — evitando
  downgrades, reinstalações e a necessidade de reiniciar a sessão.
- **nibabel:** biblioteca de leitura/escrita de arquivos **NIfTI** (`.nii.gz`), o formato dos
  volumes do dataset. O MONAI a utiliza internamente para carregar as imagens.

> 🔭 **Compatibilidade futura (DPC §9.2).** A v1.0 mantém o conjunto de dependências enxuto.
> Quando o projeto evoluir para **Transformers** (v2.0: UNETR, SwinUNETR), bastará acrescentar
> extras como `einops` a esta mesma célula — a estrutura da seção não muda. (No Colab atual, o
> `einops` já vem instalado, como o inventário do passo 1.4 confirma.)

In [ ]:
# Versão fixada para reprodutibilidade (DPC §6.6). Coincide com a que o Colab entrega
# pré-instalada, então a instalação é idempotente: sem downgrade, sem reiniciar a sessão.
MONAI_VERSION = "1.6.0"

if EM_COLAB:
    import subprocess, sys
    pacotes = [f"monai=={MONAI_VERSION}", "nibabel"]
    print("Garantindo dependências:", ", ".join(pacotes))
    # O PyTorch do Colab é preservado (não aparece na lista, portanto não é reinstalado).
    resultado = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *pacotes],
        capture_output=True, text=True,
    )
    if resultado.returncode == 0:
        print("Dependências prontas.")
    else:
        # Só mostramos os logs do pip se algo falhar — mantém a saída limpa no caso comum.
        print("Falha na instalação. Logs do pip:")
        print(resultado.stdout)
        print(resultado.stderr)
else:
    # Em ambiente local, presume-se que as dependências já estejam instaladas
    # (ex.: via `pip install monai==1.6.0 nibabel`). Apenas avisamos.
    print("Ambiente local: verifique manualmente que 'monai' e 'nibabel' estão instalados.")

## 1.3 · Configuração do dispositivo (GPU / CPU)

Modelos 3D exploram simultaneamente altura, largura e profundidade dos exames, preservando as
relações anatômicas — ao custo de **muito mais memória e computação** (DPC §3.4). Na prática,
isso torna a **GPU essencial** para treinar em tempo razoável.

O Colab oferece GPU, mas sua **disponibilidade é variável** (DPC §10.1). Por isso a célula
abaixo:

- seleciona a **GPU** (`cuda`) quando disponível e imprime seu nome, memória e a versão de CUDA;
- faz um **fallback gracioso para CPU**, com um **aviso explícito** de que o treinamento será
  lento — evitando que alguém treine na CPU sem perceber.

A variável `device` resultante é **reutilizada por todas as seções seguintes** (modelo, dados
e inferência serão enviados para ela).

> 💡 **Como habilitar a GPU no Colab:** menu *Ambiente de execução → Alterar o tipo de
> ambiente de execução → Acelerador de hardware → GPU*.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    nome_gpu = torch.cuda.get_device_name(0)
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU disponível: {nome_gpu}")
    print(f"   Memória total : {memoria_gb:.1f} GB")
    print(f"   Versão de CUDA: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("⚠️  Nenhuma GPU detectada — usando CPU.")
    print("    O treinamento de um modelo 3D na CPU é MUITO lento e não é recomendado.")
    print("    No Colab: Ambiente de execução → Alterar o tipo de ambiente → GPU.")

print(f"\nDispositivo selecionado: {device}")

## 1.4 · Verificação e registro de proveniência

Reprodutibilidade não é só *fixar* versões — é **registrá-las**. Cada experimento do projeto
deve poder ser reconstruído a partir de um "cartão de identidade" do ambiente: versões de
bibliotecas, dispositivo e data de execução (DPC §6.6 e §7.2).

Fazemos isso em dois passos:

1. **Inventário completo via MONAI.** Usamos o componente **oficial** `monai.config.print_config()`
   — que já lista as versões do MONAI, do PyTorch e de todas as dependências opcionais. Seguindo
   o princípio de *não reimplementar o que o MONAI já resolve* (DPC §5.4), preferimos essa função
   a montar uma verificação manual.
2. **Dicionário `AMBIENTE`.** Consolidamos os campos essenciais em uma estrutura que as seções
   futuras (treinamento, avaliação) anexarão ao registro de cada experimento.

In [ ]:
import monai

# Inventário oficial do MONAI: versões do MONAI, PyTorch, NumPy e dependências opcionais.
# (Com a instalação idempotente do passo 1.2, este import é direto — sem reiniciar a sessão.)
monai.config.print_config()

In [ ]:
import platform
import numpy as np
from datetime import datetime, timezone

# Versão deste notebook — registrada em cada experimento para rastreabilidade (DPC §6.6).
VERSAO_NOTEBOOK = "3.1-perda-desbalanceada"

# "Cartão de identidade" do ambiente — reutilizado no logging de cada experimento (DPC §6.6/§7.2).
AMBIENTE = {
    "data_execucao_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "numpy": np.__version__,
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "monai": monai.__version__,
    "dispositivo": str(device),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "em_colab": EM_COLAB,
    "versao_notebook": VERSAO_NOTEBOOK,
}

print("Registro de proveniência do ambiente")
print("-" * 44)
for chave, valor in AMBIENTE.items():
    print(f"{chave:>18} : {valor}")

## ✅ Resumo da Seção 1

O ambiente está pronto e **documentado**:

- detectamos o contexto de execução (`EM_COLAB`);
- fixamos o **MONAI 1.6.0** e garantimos o **nibabel**, reutilizando o **PyTorch do Colab**;
- selecionamos o **dispositivo** de computação (`device`), com fallback e aviso para CPU;
- registramos a **proveniência** do ambiente no dicionário `AMBIENTE`.

Essas variáveis — `device` e `AMBIENTE` — atravessam todo o notebook e são a base da
reprodutibilidade dos experimentos.

**➡️ Próxima seção — Configuração Global.** Definiremos a **semente aleatória** e o **modo
determinístico** (Python, NumPy, PyTorch e MONAI, via `set_determinism`), além dos
**hiperparâmetros globais** do pipeline. Só então começaremos a tocar nos dados.

> *"Tecnologias mudam. Frameworks evoluem. Modelos são substituídos. O rigor científico permanece."*
> — Manifesto Científico da Plataforma (DPC §11.4)

# 2 · Configuração Global

Se a Seção 1 preparou o *ambiente*, esta seção estabelece o **protocolo experimental** — o
conjunto de decisões que precisa permanecer **constante** para que os resultados sejam
comparáveis e reproduzíveis. O DPC é enfático (§5.1, §6): *"pequenas alterações no
pré-processamento, na divisão dos dados, nas métricas ou na configuração do treinamento podem
produzir diferenças substanciais nos resultados"*. Por isso essas decisões são declaradas
**uma única vez, em um só lugar**, antes de qualquer código de dados ou de modelo.

Esta seção tem duas responsabilidades:

1. **Controlar a aleatoriedade** — fixar as sementes e ativar o modo determinístico (§6.2);
2. **Centralizar os hiperparâmetros** — reunir num único objeto imutável os valores que
   definem o protocolo (§7.2, *"separação clara entre configuração e execução"*).

> 🧭 **Por que isso é o coração da comparabilidade (§9.2).** O projeto evolui em versões
> (1.0 U-Net → 1.1 CNNs modernas → 2.0 Transformers). Em todas elas, *a única variável
> deliberadamente modificada é a arquitetura*. Manter dados, pré-processamento, métricas e
> treinamento fixos aqui é o que garante que uma diferença de desempenho seja atribuível à
> **arquitetura**, e não a mudanças ocultas no protocolo.

## 2.1 · Controle de aleatoriedade

Redes neurais dependem de **acaso** em vários pontos: a inicialização dos pesos, a ordem de
embaralhamento dos dados, o *data augmentation*. Sem controlar essas fontes, duas execuções do
mesmo código produzem resultados diferentes — e um resultado que não se repete não pode ser
verificado por outro pesquisador.

A solução é **fixar uma semente (seed)**: um número que torna todas as sequências
"aleatórias" reprodutíveis. O DPC (§6.2) pede que a semente seja fixada para **Python, NumPy,
PyTorch e MONAI**, com modos determinísticos quando tecnicamente possível.

Em vez de semear cada biblioteca manualmente, usamos o componente **oficial**
`monai.utils.set_determinism()` (princípio §5.4 — não reimplementar). Com uma única chamada
ele semeia `random`, `numpy` e `torch`, e ativa o modo determinístico do cuDNN.

> ⚖️ **Uma nota honesta sobre limites (§6.2).** Mesmo com a semente fixa, pequenas diferenças
> numéricas podem surgir em GPU — algumas operações 3D não possuem implementação determinística.
> Por isso **não** forçamos `torch.use_deterministic_algorithms(True)`: isso interromperia o
> treino com erro em vez de apenas variar na última casa decimal. O DPC aceita explicitamente
> essa pequena variação de hardware como razoável.

In [ ]:
from monai.utils import set_determinism

# Semente mestra do projeto. Um valor único e documentado, reutilizado em todo o pipeline.
SEED = 0

# Semeia Python (random), NumPy e PyTorch de uma só vez e ativa o modo determinístico do cuDNN.
set_determinism(seed=SEED)

print(f"Determinismo ativado com SEED = {SEED}")
print("Sementes fixadas para: random (Python), NumPy e PyTorch; cuDNN em modo determinístico.")

## 2.2 · Hiperparâmetros globais

Aqui declaramos, num **único objeto**, todos os hiperparâmetros do protocolo experimental.
Reuni-los em um só lugar traz três benefícios diretos do DPC:

- **Comparabilidade (§5.1):** o protocolo fica visível e fixo; trocar a arquitetura nas
  versões futuras não exige tocar em nenhum destes valores.
- **Transparência (§5.1):** nada de "números mágicos" espalhados pelo código — toda constante
  tem nome e lugar.
- **Reprodutibilidade (§6.6):** este objeto é registrado junto ao experimento, permitindo
  reconstruí-lo com exatidão.

Usamos um `dataclass` **imutável** (`frozen=True`): uma vez definido, o `CONFIG` **não pode ser
alterado acidentalmente durante a execução** — uma salvaguarda alinhada ao princípio de que
cada experimento é uma unidade preservada, nunca sobrescrita (§8.3).

> 📖 **Modularidade (§5.1).** Declaramos os valores aqui, mas a *justificativa metodológica* de
> cada um pertence à sua seção: o significado de `pixdim`, `a_min/a_max` e `spatial_size` é
> explicado no **Pré-processamento (Seção 4)**; as frações `val_split`/`test_split` da divisão
> por paciente, na **Preparação do Dataset (Seção 3)**; e o cronograma de treino, na **Seção 7**.
>
> ⚠️ **Revisão crítica (§8.2).** A `learning_rate = 1e-5` e o `max_epochs = 600` são herdados
> do script de referência e mantidos para estabelecer a *baseline*. São candidatos naturais a
> ajuste futuro — mas qualquer mudança será uma **decisão registrada**, não silenciosa, para
> preservar a comparabilidade entre versões.

In [ ]:
from dataclasses import dataclass, asdict

# Protocolo experimental — constante DENTRO de uma ilha de comparabilidade (só a arquitetura
# muda). O que define o PROBLEMA (dataset, modalidade, rótulos, normalização de intensidade)
# vive em `Problema`, separado: é o que permite portar o mesmo esqueleto para outras tarefas
# sem forkar código (DPC §5.2).
@dataclass(frozen=True)
class Config:

    # — Reprodutibilidade —
    seed: int = SEED

    # — Pré-processamento espacial (§6.3) · justificado na Seção 4 —
    pixdim: tuple[float, float, float] = (1.5, 1.5, 1.0)   # reamostragem espacial (mm)
    spatial_size: tuple[int, int, int] = (128, 128, 64)    # tamanho do volume após redimensionar

    # — Dados e divisão (§6.1) · realizada na Seção 3 —
    val_split: float = 0.15     # fração do conjunto rotulado para validação (separação por paciente)
    test_split: float = 0.15    # fração do conjunto rotulado para teste (isolado até a avaliação final)
    batch_size: int = 1         # volumes 3D são grandes; lote pequeno cabe na memória da GPU
    num_workers: int = 0        # 0 = carrega no processo principal (evita picos de RAM no Colab)

    # — Treinamento (§6.4) · usado na Seção 7 —
    learning_rate: float = 1e-5        # taxa de aprendizado do otimizador
    weight_decay: float = 1e-5         # regularização L2
    max_epochs: int = 600              # número máximo de épocas (limites de tempo do Colab podem reduzir)
    val_interval: int = 1              # validar a cada N épocas
    early_stopping_patience: int = 30  # validações sem melhora antes de parar (§6.1)


# O PROBLEMA investigado — define o que muda entre ilhas de comparabilidade. A normalização de
# intensidade é específica da MODALIDADE (janela HU em CT); realocá-la aqui prepara a troca por
# z-score na futura versão de MRI, sem tocar no esqueleto.
@dataclass(frozen=True)
class Problema:
    nome: str                       # identifica a pasta de resultados
    dataset_task: str               # tarefa do Medical Segmentation Decathlon
    modalidade: str                 # "CT" | "MRI" | ...
    mapa_orig: tuple[int, ...]      # rótulos originais do dataset
    mapa_destino: tuple[int, ...]   # rótulos após remapeamento (define o espaço de classes)
    classes: tuple[str, ...]        # nomes das classes, incluindo o fundo (índice 0)
    a_min: float                    # normalização de intensidade (modalidade-específica)
    a_max: float
    b_min: float
    b_max: float
    clip: bool

    @property
    def num_classes(self) -> int:
        return len(self.classes)

    @property
    def classes_fg(self) -> tuple[str, ...]:
        return self.classes[1:]     # classes de primeiro plano (sem o fundo)


# Instâncias únicas e imutáveis usadas por todo o pipeline.
CONFIG = Config()
PROBLEMA = Problema(
    nome="figado_multiclasse",
    dataset_task="Task03_Liver",
    modalidade="CT",
    mapa_orig=(0, 1, 2),
    mapa_destino=(0, 1, 2),         # preserva fígado (1) e tumor (2) como classes distintas
    classes=("fundo", "figado", "tumor"),
    a_min=-200.0, a_max=200.0, b_min=0.0, b_max=1.0, clip=True,
)

print("Hiperparâmetros globais (CONFIG):")
print("-" * 44)
for chave, valor in asdict(CONFIG).items():
    print(f"{chave:>16} : {valor}")
print("\nProblema investigado (PROBLEMA):")
print("-" * 44)
for chave, valor in asdict(PROBLEMA).items():
    print(f"{chave:>16} : {valor}")
print(f"{'num_classes':>16} : {PROBLEMA.num_classes}")

## 2.3 · Registro do experimento

Reprodutibilidade se completa quando **ambiente** (Seção 1) e **protocolo** (esta seção) são
registrados juntos. Combinamos `AMBIENTE` e `CONFIG` num único dicionário — o "prontuário" do
experimento —, que as seções de treinamento e avaliação anexarão aos resultados salvos (§6.6,
§7.2, §8.3). Assim, qualquer execução pode ser reconstruída a partir do que ficou registrado.

In [ ]:
# Prontuário do experimento: ambiente (versões/dispositivo) + protocolo (hiperparâmetros).
REGISTRO_EXPERIMENTO = {
    "ambiente": AMBIENTE,          # definido na Seção 1
    "configuracao": asdict(CONFIG),
}

import json
print("Registro completo do experimento (a ser salvo junto aos resultados):")
print(json.dumps(REGISTRO_EXPERIMENTO, indent=2, ensure_ascii=False))

## ✅ Resumo da Seção 2

O protocolo experimental está estabelecido e registrado:

- fixamos a semente mestra (`SEED`) e ativamos o determinismo via `set_determinism` (§6.2);
- centralizamos os hiperparâmetros num `CONFIG` **imutável**, que fixa o protocolo entre
  versões e evita "números mágicos" (§5.1, §7.2);
- consolidamos ambiente + protocolo em `REGISTRO_EXPERIMENTO`, base do logging reproduzível (§6.6).

A partir daqui, todo o pipeline consome esses valores — **nenhuma seção seguinte redefine um
hiperparâmetro do protocolo**; elas apenas o utilizam.

**➡️ Próxima seção — Preparação e Exploração do Dataset.** Vamos localizar o MSD Task03_Liver,
**validar sua estrutura e integridade** (sem transformar nada ainda, §5.5) e realizar a divisão
por paciente em treino/validação/teste (§6.1), usando os `val_split`/`test_split` definidos aqui.

# 3 · Preparação e Exploração do Dataset

Esta é a camada de **Dados** (DPC §5.5). Sua responsabilidade é deliberadamente restrita:
**localizar o dataset, validar sua estrutura e integridade e organizá-lo em conjuntos** — mas
*sem transformar nada ainda*. Toda transformação (orientação, reamostragem, normalização) fica
para o Pré-processamento (Seção 4). Aqui apenas garantimos que os dados existem, estão íntegros
e corretamente pareados.

Trabalhamos com o **Medical Segmentation Decathlon — Task03_Liver** (DPC §4), um benchmark
público de CT hepática. Seguindo o princípio §6.1 — *"o dataset é mantido exatamente em sua
estrutura oficial; nenhum arquivo original é modificado"* —, esperamos a **estrutura oficial do
MSD**:

```
Task03_Liver/
├── imagesTr/     # volumes de CT rotulados   (liver_0.nii.gz, liver_1.nii.gz, ...)
├── labelsTr/     # máscaras correspondentes   (mesmos nomes de imagesTr)
├── imagesTs/     # volumes de teste SEM rótulo público
└── dataset.json  # metadados oficiais (modalidade, rótulos, contagens)
```

Nesta seção nós vamos:

1. **Montar o Google Drive** e localizar o dataset;
2. **Obter o dataset** — download *opcional e idempotente*, caso ainda não esteja no Drive;
3. **Validar a estrutura** oficial e ler os metadados (§7.1);
4. **Parear e conferir a integridade** de imagens e máscaras (§7.1);
5. **Dividir por paciente** em treino/validação/teste (§6.1);
6. **Explorar** visualmente um exemplo para confirmar que tudo faz sentido.

> ⚙️ **Uma decisão de nomenclatura.** O script de referência usava as chaves `vol`/`seg`.
> Adotamos aqui as chaves **`image`/`label`**, padrão nos exemplos oficiais do MONAI e no
> `DecathlonDataset` — o que reduz atrito ao reutilizar componentes do framework nas próximas
> seções.
>
> 🔒 **Sobre o `imagesTs`.** Como os rótulos oficiais do conjunto de teste do MSD **não são
> públicos**, não é possível calcular métricas sobre ele. Por isso (decisão registrada, §8.2)
> derivamos treino/validação/**teste** a partir do conjunto **rotulado** (`imagesTr`/`labelsTr`);
> o `imagesTs` é deixado intocado.

## 3.1 · Montagem do Google Drive e localização do dataset

O ambiente oficial é o Colab (§5.3), e a forma mais prática de disponibilizar dezenas de
gigabytes de imagens médicas é mantê-las no **Google Drive** e montá-lo no notebook. A célula
abaixo monta o Drive e define `DATA_DIR` — **o único caminho que você talvez precise ajustar**,
apontando para a raiz da pasta `Task03_Liver`.

In [ ]:
from pathlib import Path

# >>> AJUSTE AQUI, se necessário: caminho da raiz do dataset na estrutura oficial do MSD. <<<
DATA_DIR = "/content/drive/MyDrive/Task03_Liver"

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_DIR = Path(DATA_DIR)
print("Diretório do dataset:", DATA_DIR)

## 3.2 · Obtenção do dataset (opcional e idempotente)

O Task03_Liver tem **~28 GB**. A célula abaixo permite baixá-lo **uma única vez** para o seu
Drive, direto do Colab (tráfego interno do Google, bem mais rápido que baixar no seu computador
e reenviar). Ela foi escrita para ser **segura de reexecutar** (princípio de reprodutibilidade,
§6.6):

- se o dataset **já existe** em `DATA_DIR`, ela **não baixa nada**;
- o download só ocorre se você **explicitamente** definir `BAIXAR_DATASET = True` — assim,
  reexecutar o notebook nunca dispara um download de 28 GB por acidente.

Há duas fontes possíveis (escolha em `FONTE`): o **espelho AWS S3** do MONAI (`wget` direto) ou
o **Google Drive oficial** (via `gdown`, informando o ID do `Task03_Liver.tar`).

> ⚠️ **Antes de baixar:** confirme que há **~30 GB livres** no Drive e que o link/ID está
> vigente (veja *medicaldecathlon.com*). A extração para o Drive pode **demorar bastante** — é
> normal. Faça uma vez; nas próximas sessões, basta montar o Drive.
>
> 💡 O `.tar` oficial já contém a pasta `Task03_Liver/` na **estrutura oficial** — por isso ele
> é extraído no diretório *pai* de `DATA_DIR`, recriando exatamente o layout esperado (§6.1).

In [ ]:
import sys, subprocess

# --- Controles (ajuste conforme sua necessidade) ---
BAIXAR_DATASET = False    # deixe False; mude para True apenas na 1ª vez, se o dataset não estiver no Drive
FONTE = "aws"             # "aws" (espelho S3 do MONAI) ou "gdrive" (Google Drive oficial)
GDRIVE_ID = ""            # se FONTE == "gdrive": ID do arquivo Task03_Liver.tar (ver medicaldecathlon.com)

URL_AWS = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task03_Liver.tar"

# Idempotência: se já houver o conteúdo rotulado, não há o que baixar.
ja_existe = (DATA_DIR / "imagesTr").is_dir() and (DATA_DIR / "labelsTr").is_dir()

if ja_existe:
    print(f"✅ Dataset já presente em '{DATA_DIR}' — download desnecessário.")
elif not BAIXAR_DATASET:
    print("ℹ️  Dataset não encontrado e BAIXAR_DATASET=False.")
    print("    Coloque o Task03_Liver no Drive, OU defina BAIXAR_DATASET=True para baixar agora.")
else:
    tar_local = "/content/Task03_Liver.tar"
    destino = DATA_DIR.parent            # o .tar cria .../Task03_Liver/ ao ser extraído aqui
    destino.mkdir(parents=True, exist_ok=True)

    if FONTE == "aws":
        print("Baixando do espelho AWS S3:", URL_AWS)
        subprocess.run(["wget", "-q", "-O", tar_local, URL_AWS], check=True)
    elif FONTE == "gdrive":
        assert GDRIVE_ID, "Defina GDRIVE_ID (veja o link vigente em medicaldecathlon.com)."
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        print("Baixando do Google Drive (ID):", GDRIVE_ID)
        subprocess.run(["gdown", "--fuzzy", GDRIVE_ID, "-O", tar_local], check=True)
    else:
        raise ValueError("FONTE deve ser 'aws' ou 'gdrive'.")

    print(f"Extraindo para '{destino}' … (pode demorar bastante)")
    subprocess.run(["tar", "-xf", tar_local, "-C", str(destino)], check=True)
    print("✅ Download e extração concluídos.")

## 3.3 · Validação da estrutura e metadados

Antes de tocar em qualquer arquivo, confirmamos que a **estrutura oficial existe**. Em IA
médica, um erro silencioso de caminho pode contaminar todo o experimento — por isso, seguindo
o DPC §7.1, *qualquer inconsistência interrompe a execução* com uma mensagem clara, em vez de
prosseguir com dados errados.

Também lemos o `dataset.json` oficial: ele documenta a modalidade (CT), o significado dos
rótulos (`0` fundo, `1` fígado, `2` lesão) e as contagens esperadas — que usamos para conferir
o que encontramos em disco.

In [ ]:
import json

# Estrutura oficial esperada (DPC §6.1 — apenas verificamos; não modificamos nada).
imagesTr    = DATA_DIR / "imagesTr"
labelsTr    = DATA_DIR / "labelsTr"
imagesTs    = DATA_DIR / "imagesTs"
dataset_json = DATA_DIR / "dataset.json"

# Diretórios rotulados são obrigatórios; sua ausência interrompe a execução (§7.1).
obrigatorios = {"imagesTr/": imagesTr, "labelsTr/": labelsTr}
faltando = [nome for nome, caminho in obrigatorios.items() if not caminho.is_dir()]
if faltando:
    raise FileNotFoundError(
        f"Estrutura oficial do MSD não encontrada em '{DATA_DIR}'. "
        f"Faltando: {', '.join(faltando)}. Verifique o DATA_DIR e a montagem do Drive."
    )
print("✅ Estrutura validada: imagesTr/ e labelsTr/ presentes.")
if not imagesTs.is_dir():
    print("ℹ️  imagesTs/ ausente — sem impacto (não usamos o teste não rotulado do MSD).")

# Metadados oficiais (quando disponíveis).
if dataset_json.is_file():
    with open(dataset_json, encoding="utf-8") as f:
        META = json.load(f)
    print("\nMetadados oficiais (dataset.json):")
    print("   Nome        :", META.get("name"))
    print("   Modalidade  :", META.get("modality"))
    print("   Rótulos     :", META.get("labels"))
    print("   numTraining :", META.get("numTraining"))
    print("   numTest     :", META.get("numTest"))
else:
    META = None
    print("\nℹ️  dataset.json ausente — seguiremos apenas pela varredura de arquivos.")

## 3.4 · Pareamento imagem–máscara

Agora listamos os volumes e as máscaras e verificamos a **correspondência 1:1** entre eles.
Dois cuidados do DPC §7.1:

- **Ordenação estável** com `sorted(glob(...))`: garante que a *n*-ésima imagem case com a
  *n*-ésima máscara, de forma reprodutível;
- **Arquivos ocultos**: descartamos entradas iniciadas por `.` (ex.: `._liver_0.nii.gz`,
  artefatos de sistemas de arquivos) que, se não filtradas, quebrariam o pareamento.

O resultado é a lista `data_dicts` — uma lista de dicionários `{"image": ..., "label": ...}`,
o formato que os *transforms* e *datasets* do MONAI consomem diretamente.

In [ ]:
import os
from glob import glob

def listar_nii(pasta: Path) -> list[str]:
    # sorted(glob(...)) => ordenação estável e reprodutível (DPC §7.1).
    arquivos = sorted(glob(os.path.join(str(pasta), "*.nii.gz")))
    # Ignora arquivos ocultos (ex.: '._liver_0.nii.gz').
    return [a for a in arquivos if not os.path.basename(a).startswith(".")]

vol_paths = listar_nii(imagesTr)
seg_paths = listar_nii(labelsTr)

# Quantidades devem coincidir...
if len(vol_paths) != len(seg_paths):
    raise RuntimeError(
        f"Nº de imagens ({len(vol_paths)}) difere do nº de máscaras ({len(seg_paths)})."
    )

# ...e cada imagem deve ter máscara de mesmo nome.
for v, s in zip(vol_paths, seg_paths):
    if os.path.basename(v) != os.path.basename(s):
        raise RuntimeError(
            f"Imagem e máscara não correspondem: '{os.path.basename(v)}' x '{os.path.basename(s)}'."
        )

data_dicts = [{"image": v, "label": s} for v, s in zip(vol_paths, seg_paths)]
print(f"✅ {len(data_dicts)} pares imagem/máscara rotulados, com correspondência 1:1 verificada.")

## 3.5 · Integridade dos arquivos

Correspondência de nomes não basta: imagem e máscara precisam descrever o **mesmo volume
físico**. Verificamos, para cada par, que as **dimensões espaciais coincidem** (§7.1). Lemos
apenas o *cabeçalho* NIfTI (via `nibabel`, sem carregar os voxels), o que torna a checagem
rápida mesmo sobre dezenas de exames.

In [ ]:
import numpy as np
import nibabel as nib
from tqdm.auto import tqdm

problemas = []
formas = []
for par in tqdm(data_dicts, desc="Verificando integridade"):
    forma_img = nib.load(par["image"]).header.get_data_shape()
    forma_seg = nib.load(par["label"]).header.get_data_shape()
    if forma_img != forma_seg:
        problemas.append((os.path.basename(par["image"]), forma_img, forma_seg))
    formas.append(forma_img)

if problemas:
    for nome, fi, fs in problemas[:10]:
        print(f"  {nome}: imagem {fi} x máscara {fs}")
    raise RuntimeError(f"{len(problemas)} par(es) com dimensões incompatíveis imagem/máscara.")

formas = np.array([f[:3] for f in formas])
print("✅ Integridade OK — imagem e máscara com mesmas dimensões em todos os exames.")
print(f"   Cortes axiais (eixo Z): mín {formas[:, 2].min()}, "
      f"máx {formas[:, 2].max()} — variação anatômica esperada entre pacientes.")

## 3.6 · Divisão por paciente: treino / validação / teste

Chegamos a uma das decisões mais sensíveis do protocolo (§6.1). A divisão segue **separação
estrita por paciente**: exames de um mesmo indivíduo nunca aparecem em conjuntos diferentes —
o que elimina uma das formas mais comuns de **vazamento de dados (*data leakage*)** em
aplicações médicas. No Task03_Liver, cada arquivo corresponde a um paciente distinto, então
uma divisão por arquivo já satisfaz esse critério.

Usamos as proporções **70 / 15 / 15** definidas em `CONFIG` (`val_split`, `test_split`). Para
particionar, empregamos o componente **oficial** `monai.data.partition_dataset` (princípio
§5.4), com `seed=CONFIG.seed` — garantindo que a *mesma divisão* se repita em toda execução
(reprodutibilidade, §6.2). Ao final, uma verificação anti-vazamento confirma que os três
conjuntos são **disjuntos** (§7.1).

- **Treino** — otimiza os parâmetros da rede;
- **Validação** — monitora o treino e orientará a parada antecipada;
- **Teste** — permanece **isolado** até a avaliação final.

In [ ]:
from monai.data import partition_dataset

# Proporções relativas [treino, validação, teste] a partir do protocolo (CONFIG).
train_frac = 1.0 - CONFIG.val_split - CONFIG.test_split
ratios = [train_frac, CONFIG.val_split, CONFIG.test_split]

train_files, val_files, test_files = partition_dataset(
    data_dicts,
    ratios=ratios,
    shuffle=True,          # embaralha antes de dividir...
    seed=CONFIG.seed,      # ...de forma reprodutível.
)

print(f"Divisão por paciente (SEED={CONFIG.seed}) — proporções {ratios}:")
print(f"   Treino    : {len(train_files):>3} exames")
print(f"   Validação : {len(val_files):>3} exames")
print(f"   Teste     : {len(test_files):>3} exames  (isolado até a avaliação final)")

# Verificação anti-vazamento (§7.1): conjuntos disjuntos e cobertura total.
s_tr = {d["image"] for d in train_files}
s_va = {d["image"] for d in val_files}
s_te = {d["image"] for d in test_files}
assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), \
    "VAZAMENTO: há exames repetidos entre conjuntos!"
assert len(s_tr) + len(s_va) + len(s_te) == len(data_dicts), "Perda/duplicação de exames na divisão!"
print("✅ Sem sobreposição entre treino/validação/teste — separação por paciente garantida.")

## 3.7 · Exploração de um exemplo

Números não substituem o olhar. Antes de seguir, carregamos **um** exame (apenas para
inspeção — nenhuma transformação do dataset é aplicada) e olhamos um corte axial central, a
máscara e a sobreposição. Isso confirma, de forma qualitativa, que imagens e rótulos estão
coerentes.

Observe os **rótulos presentes neste exame**: o MSD marca `1` para o fígado e `2` para lesões
hepáticas — mas **nem todo paciente tem lesão**, então é normal um exame exibir apenas `[0, 1]`.
O conjunto *global* de rótulos do dataset será confirmado empiricamente na **Seção 4**, antes de
unificarmos fígado e lesão num único primeiro plano (`label > 0`), já que o escopo da v1.0 é o
**órgão** (§4.3).

> 🧭 A imagem é exibida em sua orientação **crua** (a padronização anatômica RAS acontece no
> pré-processamento). Aplicamos uma janela `[-200, 200] HU` **apenas para exibição**, o que
> realça os tecidos moles do abdome.

In [ ]:
import matplotlib.pyplot as plt
from monai.transforms import LoadImage

# Carregamento CRU de um exemplo (image_only=True devolve apenas o tensor da imagem).
carregar = LoadImage(image_only=True, ensure_channel_first=False)
exemplo = train_files[0]
img = carregar(exemplo["image"]).numpy()
seg = carregar(exemplo["label"]).numpy()

print("Exemplo:", os.path.basename(exemplo["image"]))
print("   Dimensões       :", img.shape)
print("   Intensidade (HU): min", float(img.min()), "| max", float(img.max()))
print("   Rótulos presentes:", [int(v) for v in np.unique(seg)])

z = img.shape[2] // 2                       # corte axial central
img_disp = np.clip(img[:, :, z], -200, 200)  # janela apenas para exibição
seg_disp = seg[:, :, z]

plt.figure(figsize=(13, 5))
plt.subplot(1, 3, 1)
plt.title(f"CT — corte axial z={z}"); plt.imshow(img_disp, cmap="gray"); plt.axis("off")
plt.subplot(1, 3, 2)
plt.title("Máscara (fígado + lesão)"); plt.imshow(seg_disp); plt.axis("off")
plt.subplot(1, 3, 3)
plt.title("Sobreposição"); plt.imshow(img_disp, cmap="gray")
plt.imshow(np.ma.masked_where(seg_disp == 0, seg_disp), alpha=0.5, cmap="autumn"); plt.axis("off")
plt.tight_layout(); plt.show()

## ✅ Resumo da Seção 3

A camada de dados está pronta e auditada:

- localizamos o dataset via Google Drive (`DATA_DIR`) na **estrutura oficial do MSD**, intocada (§6.1);
- **validamos** estrutura, metadados, pareamento 1:1 e integridade dimensional (§7.1);
- dividimos o conjunto rotulado em **treino / validação / teste** por paciente (70/15/15), de
  forma reprodutível e **sem vazamento** — com o teste isolado até o fim (§6.1);
- **exploramos** um exemplo, confirmando a coerência entre imagem e máscara.

As listas `train_files`, `val_files` e `test_files` (dicionários `{"image", "label"}`) são a
entrada da próxima etapa. **Nenhum voxel foi transformado até aqui.**

**➡️ Próxima seção — Pré-processamento.** Definiremos, com *transforms* do MONAI, a cadeia que
prepara cada exame para a rede: leitura, padronização de orientação (RAS), reamostragem
(`pixdim`), normalização de intensidade (`a_min/a_max`), recorte de ROI e redimensionamento
(`spatial_size`) — além do **data augmentation** aplicado *somente* ao treino (§6.3).

# 4 · Pré-processamento

Exames de CT chegam em formas, orientações, resoluções e faixas de intensidade muito diferentes
entre pacientes e protocolos. A rede, porém, espera entradas **padronizadas**. O pré-processamento
é a ponte entre esses dois mundos — e o DPC (§6.3) é claro: *todo o pré-processamento ocorre
dinamicamente no carregamento, via operadores do MONAI*. Nada é gravado em disco; as
transformações são aplicadas **exame a exame**, no momento em que cada um é lido.

Trabalhamos com *transforms de dicionário* do MONAI (os que terminam em `d`), que operam sobre o
par `{"image", "label"}` mantendo imagem e máscara **em sincronia** — uma rotação aplicada à
imagem é aplicada identicamente à máscara.

Nesta seção nós vamos:

1. **Confirmar empiricamente** o conjunto de rótulos do dataset (fecha a pendência da Seção 3);
2. Definir a **cadeia base** determinística (leitura → orientação → reamostragem → normalização → recorte → redimensionamento);
3. Definir o **data augmentation**, aplicado **somente ao treino** (§6.3);
4. **Montar** os pipelines de treino e de validação/teste;
5. **Visualizar** o efeito do pré-processamento e do augmentation.

> 🧭 **Princípio de comparabilidade (§5.1).** Todos os hiperparâmetros aqui (`pixdim`,
> `a_min/a_max`, `spatial_size`) vêm do `CONFIG` — não há números soltos. Trocar a arquitetura
> nas versões futuras não altera **nada** deste pré-processamento.

## 4.1 · Confirmação empírica dos rótulos

Na Seção 3, o exame de exemplo exibiu apenas `[0, 1]` (não tinha lesão). Antes de unificar as
classes, vale **verificar em todo o conjunto rotulado** quais valores de rótulo realmente
existem — em vez de confiar apenas no `dataset.json`. A varredura abaixo lê cada máscara uma vez
e acumula o conjunto de rótulos, além de contar quantos exames contêm lesão (rótulo `2`).

> ⏱️ É uma verificação **única** que percorre os 131 exames; pode levar alguns minutos (leitura
> a partir do Drive). Para uma checagem rápida, defina `N_SCAN` como um inteiro para amostrar.

In [ ]:
import numpy as np
import nibabel as nib
from tqdm.auto import tqdm

# N_SCAN = None varre TODOS os exames rotulados; um inteiro amostra os primeiros N (mais rápido).
N_SCAN = None
alvos = data_dicts if N_SCAN is None else data_dicts[:N_SCAN]

rotulos_globais = set()
exames_com_lesao = 0
for par in tqdm(alvos, desc="Varredura de rótulos"):
    arr = np.asarray(nib.load(par["label"]).dataobj)   # dataobj = leitura preguiçosa
    valores = {int(v) for v in np.unique(arr)}
    rotulos_globais |= valores
    if 2 in valores:
        exames_com_lesao += 1

print("Rótulos encontrados no conjunto :", sorted(rotulos_globais))
print(f"Exames com lesão (rótulo 2)     : {exames_com_lesao}/{len(alvos)}")
print("Interpretação: 0=fundo, 1=fígado, 2=tumor (lesão hepática).")
print("Decisão (v3.0, §4.3/§10.4): preservamos as TRÊS classes — segmentação multiclasse.")

## 4.2 · A cadeia base de transformações (determinística)

Esta é a sequência aplicada **igualmente** a treino, validação e teste. A ordem importa — cada
passo pressupõe o anterior:

| # | Transform (MONAI) | O que faz | Por quê |
|---|---|---|---|
| 1 | `LoadImaged` | Lê os arquivos NIfTI | Traz imagem e máscara para a memória |
| 2 | `EnsureChannelFirstd` | Garante o eixo de canal à frente `(C, H, W, D)` | Formato que o PyTorch/MONAI espera |
| 3 | `MapLabelValued` | Unifica rótulos `1` e `2` → `1` | Escopo v1.0 = fígado como órgão (§4.3) |
| 4 | `Orientationd` (RAS) | Padroniza a orientação anatômica | Torna todos os exames comparáveis no espaço |
| 5 | `Spacingd` (`pixdim`) | Reamostra para espaçamento físico uniforme | Remove diferenças de resolução entre aparelhos |
| 6 | `ScaleIntensityRanged` | Janela de HU `[-200, 200]` → `[0, 1]` | Realça tecidos moles; normaliza a intensidade |
| 7 | `CropForegroundd` | Recorta o ar ao redor do corpo | Concentra a ROI, reduz custo e desbalanceamento |
| 8 | `Resized` (`spatial_size`) | Redimensiona para `(128, 128, 64)` | Tamanho fixo exigido pela rede |

> 🔧 **Correção metodológica em relação à referência.** O script original binarizava a máscara
> **dentro do laço de treino** (`label != 0`) e redimensionava imagem e máscara com a *mesma*
> interpolação. Aqui, (a) a unificação de rótulos é um **transform explícito** (`MapLabelValued`),
> e (b) o `Resized` usa **`area`** para a imagem e **`nearest`** para a máscara — pois interpolar
> uma máscara discreta com suavização produziria rótulos inválidos (valores fracionários).

> 📐 **Decisão registrada — `Spacingd` + `Resized` para tamanho fixo (§8.2).** Note que o
> `Spacingd` padroniza a *resolução física* (mm/voxel), mas o `Resized` para um *shape fixo* logo
> em seguida faz com que cada exame termine com um tamanho de voxel físico **diferente** (fígados
> de extensões distintas cabem na mesma grade 128×128×64). Mantemos o `Resized` deliberadamente:
> ele encolhe o **órgão inteiro** para caber no orçamento de memória da GPU do Colab — um *pad/crop*
> para tamanho fixo, ao contrário, cortaria boa parte do fígado (o eixo Z teria apenas ~64 mm de
> campo de visão). A única consequência é sobre **métricas de distância**: por isso o **HD95 é
> calculado em mm, informando o espaçamento real de cada exame** (Seção 9.1). Dice e IoU, sendo
> razões, não são afetados.

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, MapLabelValued, Orientationd,
    Spacingd, ScaleIntensityRanged, CropForegroundd, Resized, EnsureTyped,
    RandFlipd, RandAffined, RandShiftIntensityd, RandGaussianNoised,
)

CHAVES = ["image", "label"]

# Cadeia BASE (determinística) — idêntica para treino, validação e teste.
transforms_base = [
    LoadImaged(keys=CHAVES),
    EnsureChannelFirstd(keys=CHAVES),
    # Remapeamento de rótulos definido pelo PROBLEMA (Seção 2). Na v3.0 (multiclasse) o mapa é
    # identidade (0,1,2): preserva fundo, fígado e tumor como classes distintas.
    MapLabelValued(keys=["label"],
                   orig_labels=list(PROBLEMA.mapa_orig),
                   target_labels=list(PROBLEMA.mapa_destino)),
    Orientationd(keys=CHAVES, axcodes="RAS"),
    Spacingd(keys=CHAVES, pixdim=CONFIG.pixdim, mode=("bilinear", "nearest")),
    # Normalização de intensidade específica da modalidade (janela HU em CT), vinda do PROBLEMA.
    ScaleIntensityRanged(
        keys=["image"], a_min=PROBLEMA.a_min, a_max=PROBLEMA.a_max,
        b_min=PROBLEMA.b_min, b_max=PROBLEMA.b_max, clip=PROBLEMA.clip,
    ),
    CropForegroundd(keys=CHAVES, source_key="image"),
    # Imagem: 'area' (bom para reduzir escala). Máscara: 'nearest' (preserva rótulos discretos).
    Resized(keys=CHAVES, spatial_size=CONFIG.spatial_size, mode=("area", "nearest")),
]
print(f"Cadeia base definida: {len(transforms_base)} transformações determinísticas.")

## 4.3 · Data augmentation (somente no treino)

Para melhorar a generalização, ampliamos artificialmente a variedade do **treino** com pequenas
perturbações aleatórias (§6.3). Um cuidado metodológico central: o augmentation é aplicado
**exclusivamente ao conjunto de treino** — validação e teste permanecem **inalterados**, para
que suas métricas reflitam desempenho real, não versões distorcidas dos dados.

As transformações são **suaves** (o DPC pede "pequenas deformações e variações"):

- `RandFlipd` — espelhamentos ocasionais;
- `RandAffined` — rotações (~±6°), pequenos deslocamentos e variações de escala (±10%), cobrindo "rotações, deslocamentos e pequenas deformações";
- `RandShiftIntensityd` / `RandGaussianNoised` — variações de intensidade e ruído leve, simulando diferenças de aquisição.

> 🎲 Como a semente global foi fixada na Seção 2, o augmentation é **aleatório porém
> reprodutível**: a mesma execução gera a mesma sequência de perturbações.

In [ ]:
# Data augmentation — aplicado SOMENTE ao treino (§6.3). Máscara acompana com interpolação 'nearest'.
transforms_aug = [
    RandFlipd(keys=CHAVES, prob=0.20, spatial_axis=0),
    RandAffined(
        keys=CHAVES, prob=0.30,
        rotate_range=(0.10, 0.10, 0.10),   # radianos (~±5.7°) por eixo
        translate_range=(5, 5, 5),         # voxels
        scale_range=(0.10, 0.10, 0.10),    # ±10%
        mode=("bilinear", "nearest"),
        padding_mode="border",
    ),
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.30),
    RandGaussianNoised(keys=["image"], prob=0.15, std=0.01),
]
print(f"Data augmentation definido: {len(transforms_aug)} transformações (apenas no treino).")

## 4.4 · Montagem dos pipelines

Compomos as cadeias finais. A diferença entre treino e validação/teste é **apenas** o bloco de
augmentation — a espinha dorsal determinística é a mesma, garantindo que os três conjuntos sejam
processados de forma consistente. O `EnsureTyped` ao final garante tensores prontos para a rede.

In [ ]:
final = [EnsureTyped(keys=CHAVES)]

train_transforms = Compose(transforms_base + transforms_aug + final)
val_transforms   = Compose(transforms_base + final)   # determinístico: sem augmentation
test_transforms  = val_transforms                     # teste = mesmo pré-processamento da validação

print(f"train_transforms      : {len(train_transforms.transforms)} passos (base + augmentation)")
print(f"val/test_transforms   : {len(val_transforms.transforms)} passos (somente base)")

## 4.5 · Visualização do resultado

Aplicamos a cadeia **de validação** (determinística) a um exame e conferimos: a forma final deve
ser `(128, 128, 64)`, a intensidade deve estar em `[0, 1]` e a máscara deve conter apenas `{0, 1}`.
Em seguida, aplicamos a cadeia **de treino** ao mesmo exame algumas vezes, para *ver* o
data augmentation em ação (cada passagem produz uma variação diferente).

In [ ]:
import matplotlib.pyplot as plt
from monai.data import Dataset, DataLoader
from monai.utils import first

# Um exemplo passado pela cadeia determinística (validação).
ds_demo = Dataset(data=val_files[:1], transform=val_transforms)
amostra = first(DataLoader(ds_demo, batch_size=1))
img = amostra["image"][0, 0].cpu().numpy()   # (H, W, D)
seg = amostra["label"][0, 0].cpu().numpy()

print("Após o pré-processamento:")
print("   Forma da imagem :", img.shape, " (esperado:", tuple(CONFIG.spatial_size), ")")
print(f"   Intensidade     : min {img.min():.3f} | max {img.max():.3f}  (esperado ~[0, 1])")
print("   Rótulos         :", [int(v) for v in np.unique(seg)], " (esperado: [0, 1])")

z = img.shape[2] // 2
plt.figure(figsize=(12, 4.5))
plt.subplot(1, 3, 1); plt.title(f"Imagem normalizada (z={z})")
plt.imshow(img[:, :, z], cmap="gray"); plt.axis("off")
plt.subplot(1, 3, 2); plt.title("Máscara (fígado)")
plt.imshow(seg[:, :, z]); plt.axis("off")
plt.subplot(1, 3, 3); plt.title("Sobreposição")
plt.imshow(img[:, :, z], cmap="gray")
plt.imshow(np.ma.masked_where(seg[:, :, z] == 0, seg[:, :, z]), alpha=0.5, cmap="autumn"); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# O mesmo exame, três passagens pela cadeia de TREINO: cada acesso re-aplica o augmentation aleatório.
ds_aug = Dataset(data=train_files[:1], transform=train_transforms)

plt.figure(figsize=(12, 4.5))
for j in range(3):
    exemplo_aug = ds_aug[0]
    a_img = exemplo_aug["image"][0].cpu().numpy()
    zz = a_img.shape[2] // 2
    plt.subplot(1, 3, j + 1); plt.title(f"Treino — augmentation #{j + 1}")
    plt.imshow(a_img[:, :, zz], cmap="gray"); plt.axis("off")
plt.tight_layout(); plt.show()

## ✅ Resumo da Seção 4

O pré-processamento está definido, inteiramente com componentes oficiais do MONAI:

- **confirmamos** empiricamente os rótulos do dataset (`{0, 1, 2}`) e a decisão de unificá-los;
- definimos a **cadeia base** determinística (orientação, reamostragem, normalização, recorte, redimensionamento), parametrizada pelo `CONFIG`;
- definimos o **data augmentation** suave, aplicado **somente ao treino** (§6.3);
- montamos `train_transforms` e `val_transforms`/`test_transforms`, e **visualizamos** o resultado.

Nenhum arquivo foi gravado: tudo é aplicado dinamicamente no carregamento. Os objetos
`train_transforms`, `val_transforms` e `test_transforms` são a entrada da próxima etapa.

**➡️ Próxima seção — DataLoaders.** Envolveremos os arquivos e as transformações em *datasets* e
*DataLoaders* do MONAI (com `PersistentDataset` para cachear em disco), controlando `batch_size`,
`num_workers` e embaralhamento — prontos para alimentar o treinamento.

# 5 · DataLoaders

Já temos *o que* carregar (as listas `train/val/test_files`) e *como* transformar (os
`*_transforms`). Falta o mecanismo que **entrega os dados à rede em lotes**, de forma eficiente:
os *datasets* e *DataLoaders*.

Duas peças do MONAI (§5.4 — reutilizar o framework em vez de reimplementar):

- **`PersistentDataset`** — aplica a parte *determinística* das transformações **uma única vez** e
  guarda o resultado em **disco** (não em memória). Como a reamostragem e o redimensionamento de
  volumes 3D são caros — e carregar o exame original em resolução cheia consome muita RAM —, essa
  escolha é decisiva no Colab: o processamento pesado ocorre **uma vez**, e as épocas seguintes
  leem tensores pequenos do disco. O cache vai **até a primeira transformação aleatória**; o *data
  augmentation* continua sendo aplicado a cada época, preservando sua variabilidade.
- **`DataLoader`** — agrupa os exames em lotes, embaralha (só no treino) e monta os tensores.
  Usamos o `DataLoader` do MONAI, que sabe agrupar `MetaTensor` corretamente.

> 🛠️ **Decisão registrada — `PersistentDataset` em vez de `CacheDataset` (§8.2).** O
> `CacheDataset` mantém tudo em **RAM**, o que estoura a memória em planos do Colab com pouca RAM
> (o carregamento de volumes 512×512×~900 em resolução cheia gera picos de vários GB). O
> `PersistentDataset` resolve isso guardando o resultado em **disco**, com dois ganhos: **RAM
> baixa e estável** e — como o cache fica no **Google Drive** — **sobrevive a reinícios de
> sessão** (o pré-processamento pesado não precisa ser refeito a cada sessão do Colab).

Nesta seção nós vamos:

1. Construir os **datasets** (treino e validação com cache em disco; teste sob demanda) e **pré-aquecer** o cache;
2. Construir os **DataLoaders** (embaralhamento só no treino);
3. **Verificar um lote** — dimensões, tipos e faixas (§7.1).

## 5.1 · Datasets (cache em disco)

Envolvemos cada divisão em um dataset:

- **treino** e **validação** usam `PersistentDataset` — o custo determinístico é pago uma vez e
  gravado em disco (no Drive);
- **teste** usa um `Dataset` simples: é percorrido apenas na avaliação final.

Em seguida, **pré-aquecemos** o cache: percorremos cada exame uma vez, de forma **sequencial**
(um volume por vez → RAM baixa), para que o processamento pesado aconteça agora, com barra de
progresso, e não durante o treino. A operação é **idempotente** — itens já cacheados são pulados,
então reexecutar (ou retomar após um reinício) é seguro e rápido.

> ⏱️ **Primeira vez.** O pré-aquecimento lê e reamostra todos os volumes uma única vez — **pode
> levar alguns minutos**. Depois disso, aqui e nas próximas sessões, o cache do Drive é reutilizado.
>
> 🔒 **Cache versionado (§8.2).** O `PersistentDataset` indexa o cache pelo *arquivo* de entrada,
> não pelos parâmetros de pré-processamento. Para nunca reutilizar um cache obsoleto por engano,
> incluímos no caminho do `CACHE_DIR` um **hash** dos parâmetros determinísticos (`pixdim`, janela
> de intensidade, `spatial_size`): se qualquer um mudar, um novo diretório de cache é criado e
> reconstruído automaticamente.

In [ ]:
import hashlib
from pathlib import Path
from monai.data import PersistentDataset, Dataset, DataLoader
from monai.utils import first
from tqdm.auto import tqdm

# O PersistentDataset indexa o cache pelo ARQUIVO de entrada, não pelos parâmetros de
# pré-processamento. Para evitar reuso silencioso de cache obsoleto quando o pré-processamento
# mudar, versionamos o diretório com um hash dos parâmetros determinísticos relevantes.
_chave_preproc = repr((
    CONFIG.pixdim, PROBLEMA.a_min, PROBLEMA.a_max, PROBLEMA.b_min, PROBLEMA.b_max,
    PROBLEMA.clip, tuple(CONFIG.spatial_size),
))
HASH_PREPROC = hashlib.sha1(_chave_preproc.encode()).hexdigest()[:8]

# Cache em disco no Drive (persiste entre sessões). Muda automaticamente se o pré-processamento mudar.
CACHE_DIR = Path("/content/drive/MyDrive/U-Net-Liver/cache_preprocessado") / HASH_PREPROC
(CACHE_DIR / "treino").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "val").mkdir(parents=True, exist_ok=True)
print("Cache de pré-processamento (hash", HASH_PREPROC + "):", CACHE_DIR)

# Treino e validação: resultado determinístico gravado em disco; augmentation continua por época.
train_ds = PersistentDataset(data=train_files, transform=train_transforms, cache_dir=CACHE_DIR / "treino")
val_ds   = PersistentDataset(data=val_files,   transform=val_transforms,   cache_dir=CACHE_DIR / "val")
# Teste: processado sob demanda (usado apenas na avaliação final).
test_ds  = Dataset(data=test_files, transform=test_transforms)

print(f"Datasets criados — treino: {len(train_ds)} | validação: {len(val_ds)} | teste: {len(test_ds)}")

# Pré-aquecimento sequencial do cache (um volume por vez → RAM baixa; idempotente).
for ds, nome in [(val_ds, "validação"), (train_ds, "treino")]:
    for i in tqdm(range(len(ds)), desc=f"Pré-aquecendo cache ({nome})"):
        _ = ds[i]
print("✅ Cache em disco pronto.")

## 5.2 · DataLoaders

Agora os *loaders*. Três decisões metodológicas:

- **`shuffle=True` apenas no treino** — embaralhar a cada época evita que a rede aprenda a
  *ordem* dos exames; validação e teste não são embaralhados, para resultados estáveis e
  reproduzíveis (§7.1);
- **`batch_size`** vem do `CONFIG` (volumes 3D são grandes; `1` cabe confortavelmente na T4);
- **`num_workers`** vem do `CONFIG`. O padrão é **`0`** (carregamento no processo principal):
  como o cache em disco já entrega tensores pequenos e prontos, não há ganho em paralelizar — e
  isso **evita os picos de RAM** que derrubam o kernel em planos com pouca memória. `pin_memory`
  acelera a transferência para a GPU quando ela está disponível.

In [ ]:
usar_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds, batch_size=CONFIG.batch_size, shuffle=True,
    num_workers=CONFIG.num_workers, pin_memory=usar_gpu,
)
# Validação e teste percorrem um volume por vez, sem embaralhar.
val_loader = DataLoader(
    val_ds, batch_size=1, shuffle=False,
    num_workers=CONFIG.num_workers, pin_memory=usar_gpu,
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

print(f"DataLoaders prontos — treino: {len(train_loader)} lotes/época "
      f"(batch_size={CONFIG.batch_size}, shuffle=True)")
print(f"                      validação: {len(val_loader)} | teste: {len(test_loader)} (shuffle=False)")

## 5.3 · Verificação de um lote

Antes de treinar, inspecionamos **um lote real** — a última barreira de qualidade antes da rede
(§7.1). Confirmamos: as dimensões `(B, C, H, W, D)`, os tipos, a faixa de intensidade em `[0, 1]`,
os rótulos em `{0, 1}` e a **ausência de valores inválidos** (`NaN`). Qualquer inconsistência
aqui interromperia a execução antes de comprometer o treino.

In [ ]:
lote = first(train_loader)
img, seg = lote["image"], lote["label"]

print("Formato do lote:")
print("   image :", tuple(img.shape), "|", img.dtype)
print("   label :", tuple(seg.shape), "|", seg.dtype)
print(f"   image  min/max : {float(img.min()):.3f} / {float(img.max()):.3f}")
print("   label valores  :", [int(v) for v in seg.unique()])

# Barreiras de qualidade (§7.1): interrompem se algo estiver errado.
assert img.shape[0] == CONFIG.batch_size, "batch_size inesperado."
assert img.shape[1] == 1 and seg.shape[1] == 1, "Esperado 1 canal em imagem e máscara."
assert tuple(img.shape[2:]) == tuple(CONFIG.spatial_size), "Dimensões espaciais divergem do CONFIG."
assert not torch.isnan(img).any(), "Há NaN na imagem!"
assert set(int(v) for v in seg.unique()).issubset({0, 1}), "Rótulo fora de {0, 1}!"
print("\n✅ Lote válido: dimensões, canais, tipos e faixas coerentes; sem NaN.")

## ✅ Resumo da Seção 5

O fluxo de dados está completo e verificado:

- **datasets** com `PersistentDataset` (treino/validação, cache em disco no Drive) e `Dataset` (teste);
- **DataLoaders** com embaralhamento **só no treino**, `batch_size`/`num_workers` do protocolo e `pin_memory` quando há GPU;
- um **lote real inspecionado** — forma `(B, 1, 128, 128, 64)`, intensidade `[0, 1]`, rótulos `{0, 1}`, sem `NaN`.

Os objetos `train_loader`, `val_loader` e `test_loader` alimentarão as próximas etapas.

**➡️ Próxima seção — Definição da Arquitetura.** Construiremos a **U-Net 3D** do MONAI — a
*baseline* da v1.0 (§3.6) —, com sua justificativa e a estrutura pensada para permitir, nas
versões futuras, trocar apenas a arquitetura sem tocar no restante do pipeline (§9.2).

# 6 · Definição da Arquitetura (fábrica de modelos)

A **fábrica de modelos** `criar_modelo(nome)` e o seletor `ARQUITETURA` seguem **idênticos à
v3.0** — que por sua vez os herdou intactos da v2.0. A v3.1 não toca na arquitetura: o eixo desta
versão é a **função de perda** (Seção 7.1). O número de canais de saída continua derivado do
`PROBLEMA`: `NUM_CLASSES = PROBLEMA.num_classes` (**3** — fundo, fígado, tumor).

> ⚓ **Âncora desta versão: `unet`.** As cinco arquiteturas permanecem disponíveis na fábrica, mas
> o *experimento* da v3.1 roda sobre **uma única** delas, variando a perda. A pergunta em jogo —
> *"uma perda ciente de desbalanceamento tira o tumor do zero?"* — responde-se com uma arquitetura;
> a grade cheia (5 arquiteturas × 3 perdas = 15 treinos de até 600 épocas) é inviável no Colab e
> **confundiria os dois eixos**. Escolhemos a **U-Net 3D**, a *baseline* da v1.0: é a mais barata e
> estável, e é a referência do DPC (§3.6). Reconstruir a ilha com as outras quatro, usando a perda
> vencedora, é o **passo seguinte** — não este.

**As arquiteturas disponíveis** (todas do MONAI):

- **Convolucionais** — **`UNet`** (baseline histórica), **`SegResNet`** e **`DynUNet`**.
- **`UNETR`** — *Transformer* puro no encoder (ViT, autoatenção global) + decoder convolucional.
- **`SwinUNETR`** — híbrido **CNN–Transformer**: encoder Swin (atenção em janelas) + decoder U-Net.

> 🧩 **Tamanho de entrada.** Inalterado: `spatial_size = (128, 128, 64)` continua satisfazendo o
> *patch* do UNETR (÷16) e as janelas do SwinUNETR (÷32).

> ▶️ **Fluxo de execução.** Mantenha `ARQUITETURA` fixa e rode o notebook **inteiro, uma vez por
> `PERDA`** (Seção 7.1). Cada execução salva em
> `resultados/figado_multiclasse/<arquitetura>__<perda>/`; a **Seção 11** compara os experimentos.
> O cache de pré-processamento (Seção 5) é o mesmo — mais um motivo pelo qual trocar só a perda
> mantém as execuções rigorosamente pareadas.

Nesta seção nós vamos:

1. Conhecer a **fábrica de modelos** e selecionar a **arquitetura**;
2. **Construir** o modelo (com `NUM_CLASSES = 3`) e enviá-lo ao dispositivo;
3. **Verificar** a forma da saída com uma passagem de teste (§7.1).

## 6.1 · Seletor de arquitetura e fábrica de modelos

O seletor `ARQUITETURA` (`"unet"`, `"segresnet"`, `"dynunet"`, `"unetr"` ou `"swin_unetr"`)
determina qual rede será construída, treinada e avaliada nesta execução. A função `criar_modelo(nome)` concentra a
definição de cada arquitetura, todas com **os mesmos `in_channels`/`out_channels`** (definidos pela
*tarefa*, não pela arquitetura) e, sempre que aplicável, **`InstanceNorm`** — a normalização
adequada a `batch_size=1` que adotamos na v1.0 (§8.2), mantida aqui para todas as arquiteturas por
coerência de protocolo.

In [ ]:
from monai.networks.nets import UNet, SegResNet, DynUNet, UNETR, SwinUNETR
from monai.networks.layers import Norm

# >>> Arquitetura desta execução:
#     "unet" | "segresnet" | "dynunet" | "unetr" | "swin_unetr" <<<
# ÂNCORA da v3.1: mantida FIXA em "unet" (baseline da v1.0) enquanto a PERDA varia (Seção 7.1).
# Trocar as duas ao mesmo tempo confundiria os eixos e invalidaria a atribuição causal (§9.2).
ARQUITETURA = "unet"

IN_CHANNELS = 1                       # CT: um único canal de intensidade
NUM_CLASSES = PROBLEMA.num_classes    # derivado do espaço de classes do PROBLEMA (Seção 2)


def criar_modelo(nome: str):
    # Fábrica de modelos: a única coisa que varia entre as versões (§9.2).
    nome = nome.lower()
    if nome == "unet":
        # Baseline da v1.0 — preservada exatamente.
        return UNet(
            spatial_dims=3, in_channels=IN_CHANNELS, out_channels=NUM_CLASSES,
            channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
            num_res_units=2, norm=Norm.INSTANCE,
        )
    if nome == "segresnet":
        # U-Net residual (estilo BraTS). init_filters modesto para caber na T4.
        return SegResNet(
            spatial_dims=3, in_channels=IN_CHANNELS, out_channels=NUM_CLASSES,
            init_filters=16, norm="instance",
        )
    if nome == "dynunet":
        # U-Net dinâmica (estilo nnU-Net); a 1ª stride é 1, as demais reduzem a resolução.
        return DynUNet(
            spatial_dims=3, in_channels=IN_CHANNELS, out_channels=NUM_CLASSES,
            kernel_size=[3, 3, 3, 3, 3], strides=[1, 2, 2, 2, 2],
            upsample_kernel_size=[2, 2, 2, 2], norm_name="instance", res_block=True,
        )
    if nome == "unetr":
        # Transformer puro no encoder (ViT) + decoder convolucional.
        # img_size FIXO = spatial_size (128,128,64), divisível por 16 (patch do ViT).
        # hidden_size (768) é divisível por num_heads (12), como a API do MONAI exige.
        return UNETR(
            in_channels=IN_CHANNELS, out_channels=NUM_CLASSES,
            img_size=CONFIG.spatial_size, feature_size=16, hidden_size=768,
            mlp_dim=3072, num_heads=12, proj_type="conv",
            norm_name="instance", res_block=True, spatial_dims=3,
        )
    if nome == "swin_unetr":
        # Híbrido CNN-Transformer: encoder Swin (atenção em janelas) + decoder U-Net.
        # MONAI 1.6 dispensa img_size; exige dims divisíveis por 32 (128,128,64 ok).
        # feature_size=24 (padrão) mantém o consumo de memória viável na T4 do Colab.
        return SwinUNETR(
            in_channels=IN_CHANNELS, out_channels=NUM_CLASSES,
            feature_size=24, norm_name="instance", spatial_dims=3,
        )
    raise ValueError(
        f"Arquitetura desconhecida: '{nome}'. "
        "Use 'unet', 'segresnet', 'dynunet', 'unetr' ou 'swin_unetr'."
    )

print("Fábrica pronta. Arquiteturas disponíveis: unet, segresnet, dynunet, unetr, swin_unetr.")
print("Arquitetura selecionada nesta execução:", ARQUITETURA)

## 6.2 · Construção do modelo selecionado

Instanciamos a arquitetura escolhida e a enviamos ao `device` (Seção 1). O número de parâmetros
treináveis já dá uma primeira noção do "tamanho" de cada rede — um dos fatores de custo que a
comparação levará em conta (§10.4 do DPC: desempenho **e** custo importam).

In [ ]:
model = criar_modelo(ARQUITETURA).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Arquitetura '{ARQUITETURA}' construída e enviada para: {device}")
print(f"Parâmetros treináveis: {n_params:,}")

## 6.3 · Verificação da forma da saída

Antes de treinar, confirmamos que a rede produz **a forma esperada** (§7.1): para uma entrada
`(1, 1, 128, 128, 64)`, a saída deve ter `NUM_CLASSES` canais na mesma resolução espacial —
`(1, 2, 128, 128, 64)`. Cada canal de saída corresponde a uma classe (fundo, fígado); a conversão
em probabilidades e máscara final ocorre na inferência/avaliação. Usamos um tensor aleatório só
para testar as dimensões, sem gradientes.

In [ ]:
model.eval()
with torch.no_grad():
    entrada_teste = torch.randn(1, IN_CHANNELS, *CONFIG.spatial_size, device=device)
    saida_teste = model(entrada_teste)

print("Forma de entrada :", tuple(entrada_teste.shape))
print("Forma de saída   :", tuple(saida_teste.shape),
      f"(esperado: (1, {NUM_CLASSES}, {', '.join(map(str, CONFIG.spatial_size))}))")

assert tuple(saida_teste.shape) == (1, NUM_CLASSES, *CONFIG.spatial_size), \
    "A forma da saída não corresponde ao esperado!"
print(f"\n✅ Arquitetura coerente: saída com {NUM_CLASSES} canais na resolução de entrada.")

# Libera a memória do teste antes de seguir.
del entrada_teste, saida_teste
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## ✅ Resumo da Seção 6

A arquitetura desta execução está pronta:

- a **fábrica de modelos** (`criar_modelo`) reúne cinco arquiteturas do MONAI — **U-Net**,
  **SegResNet**, **DynUNet**, **UNETR** e **SwinUNETR** —, selecionáveis por `ARQUITETURA`;
- construímos a rede escolhida com `in_channels`/`out_channels` **idênticos** (definidos pela
  tarefa) e `InstanceNorm`, isolando a arquitetura como **única variável** (§9.2);
- **verificamos** a forma da saída `(1, 3, 128, 128, 64)` com uma passagem de teste (§7.1).

O objeto `model` (e as constantes `IN_CHANNELS`, `NUM_CLASSES`, `ARQUITETURA`) alimentará o
treinamento. **Nesta versão a arquitetura fica fixa** (`unet`): quem varia é a `PERDA`, e é uma
execução do notebook para cada perda antes da Seção 11.

**➡️ Próxima seção — Treinamento.** É lá que mora **o eixo da v3.1**: a **fábrica de perdas**
`criar_perda()`. Definiremos a **perda**, o **otimizador** e o **laço de treino por épocas** com
validação periódica, *early stopping* pela métrica de validação e salvamento do melhor modelo —
registrando todo o histórico (§6.4).

# 7 · Treinamento

Todas as peças estão prontas: dados (`train_loader`, `val_loader`), pré-processamento e o modelo
(`model`). Agora **ensinamos a rede a segmentar fígado e tumor**. O fluxo segue o DPC (§6.4):

> 🎯 **É aqui que a v3.1 age.** Esta é a única seção com mudança de comportamento em relação à
> v3.0: a perda deixa de ser uma constante e passa a ser **escolhida** (§7.1), e o diretório de
> resultados passa a ser chaveado também pela perda (§7.3). O laço de treino, o otimizador e a
> métrica são **idênticos** aos da v3.0.

1. definir **função de perda** e **otimizador**;
2. treinar **por épocas**, com **validação periódica**;
3. **salvar o melhor modelo** e registrar todo o histórico.

Durante o treino monitoramos: valor da **perda**, **Dice de validação**, **tempo por época** e
**consumo de memória** — os indicadores que o DPC pede acompanhar (§6.4).

> 🧪 **Métrica principal: Dice (§6.5).** O *Dice Similarity Coefficient* mede a sobreposição
> entre a máscara prevista e a de referência (0 = nenhuma, 1 = perfeita). É a métrica que guia o
> salvamento do melhor modelo e a parada antecipada.

Nesta seção nós vamos:

1. Definir **perda**, **métrica** e **otimizador**;
2. Preparar o **diretório de resultados** e registrar o experimento;
3. Definir e executar o **laço de treino** (com AMP, validação, *early stopping* e salvamento);
4. **Visualizar** as curvas de aprendizado.

## 7.1 · Função de perda e métrica — **o eixo isolado da v3.1**

Na v3.0, a perda era uma constante: `DiceCELoss`. Aquele notebook registrou explicitamente por quê
— *"isolamos o efeito da mudança de espaço de rótulos; perdas cientes de desbalanceamento ficam
para uma v3.1"*. O resultado cobrou o preço previsto: **Dice de tumor = 0,000**. Esta versão paga
essa dívida, e **só** essa.

### O diagnóstico

A `DiceCELoss` soma dois termos, e **os dois são cegos ao desbalanceamento** neste regime:

- o termo **Dice** é robusto ao desbalanceamento *fundo × primeiro plano*, mas trata as classes de
  primeiro plano **de igual para igual** — e o tumor é ordens de grandeza menor que o fígado;
- o termo **Cross-Entropy** é uma média sobre **todos os voxels**. Como a esmagadora maioria deles
  é fundo ou miolo de fígado — voxels **fáceis**, já classificados com alta confiança —, eles
  dominam a soma. O gradiente que sobra para os poucos voxels de tumor é desprezível.

O resultado é um mínimo local perfeitamente racional para o otimizador e inútil para a clínica:
**prever "não há tumor" em lugar nenhum** custa quase nada na perda.

### A fábrica de perdas

Em vez de trocar uma constante por outra, a v3.1 **parametriza** a perda — mesmo movimento que a
v3.0 fez com o objeto `Problema`, e mesma forma da fábrica `criar_modelo()` da Seção 6. Três
opções, com o racional de cada uma registrado no código (§8.2):

| `PERDA` | Perda (MONAI) | Papel nesta versão |
|---|---|---|
| `dice_ce` | `DiceCELoss` | **Controle** — a perda da v3.0, preservada |
| `dice_focal` | `DiceFocalLoss` | **Principal** — Dice idêntico + termo *focal* no lugar da CE |
| `generalized_dice` | `GeneralizedDiceLoss` | **Alternativa registrada** — pondera classes por 1/volume² |

**Por que `DiceFocalLoss` é a escolha principal.** O diff contra a `DiceCELoss` é **um único
termo**: o componente Dice permanece *idêntico*, e só a Cross-Entropy vira **Focal**. A Focal Loss
multiplica a CE por `(1 − p)^γ`: voxels já bem classificados (`p → 1`) têm sua contribuição
esmagada, e o gradiente se **concentra nos difíceis** — bordas e, sobretudo, tumor. Com `γ = 2,0`
(valor de referência da literatura), um voxel previsto com 90% de confiança pesa ~100× menos que
antes. Manter o termo Dice intacto é o que preserva a **atribuição causal**: se o resultado mudar,
mudou por causa da ponderação por dificuldade, e não por uma reformulação geral da perda.

**Por que a `GeneralizedDiceLoss` não é a primeira escolha.** Ela pondera cada classe por
`1/volume²`, o que é mais agressivo — e atraente à primeira vista. Mas com **`batch_size = 1`** e
**muitos volumes sem tumor no *ground truth***, o volume da classe rara no lote é **zero** e o peso
`1/0²` degenera. O treino ficaria numericamente instável, e um resultado ruim seria **ambíguo**:
"a perda não resolve" ou "a perda explodiu"? Ela permanece disponível e documentada, para quem
quiser testá-la — não como o braço principal.

**Por que o controle continua na fábrica.** Comparar a v3.1 contra os números da v3.0 significaria
comparar execuções diferentes, em sessões diferentes. Mantendo `dice_ce` disponível, o braço de
controle roda **agora**, com a mesma `SEED`, o mesmo *split* e o mesmo cache de pré-processamento —
comparação pareada de verdade (DPC §5.1, *Comparabilidade*).

### O que **não** muda

A **métrica** é deliberadamente a mesma da v3.0: `DiceMetric(include_background=False,
reduction="mean_batch")`, reportando Dice **por classe** (fígado, tumor). Mudar a métrica junto com
a perda tornaria o resultado ininterpretável — não se altera a régua no mesmo experimento em que se
altera o objeto medido. Pelo mesmo motivo, o **critério de seleção do melhor modelo** continua
sendo a **média foreground** do Dice de validação.

> ⚠️ **O que esta versão *não* corrige.** O `spatial_size = (128, 128, 64)` redimensiona o volume
> inteiro: lesões pequenas podem sobrar com **pouquíssimos voxels**, e nenhuma função de perda
> recupera informação que o pré-processamento já destruiu. Se o Dice de tumor continuar ≈ 0 mesmo
> com `dice_focal`, a conclusão é **positiva e informativa**: a causa dominante não é a perda, e a
> **v3.2** (amostragem focada na lesão) passa a ser uma hipótese com evidência por trás.

In [ ]:
from monai.losses import DiceCELoss, DiceFocalLoss, GeneralizedDiceLoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Compose

# >>> Escolha a PERDA desta execução — o EIXO ISOLADO da v3.1:
#     "dice_ce" (controle = v3.0) | "dice_focal" (principal) | "generalized_dice" (alternativa) <<<
PERDA = "dice_focal"

# Catálogo de perdas: classe do MONAI + hiperparâmetros EXPLÍCITOS. Os kwargs são serializados
# junto aos resultados (§6.6/§7.2) — nenhum valor relevante fica implícito no código (§5.1).
PERDAS_DISPONIVEIS = {
    # CONTROLE — exatamente a perda da v3.0. Preservada na fábrica para que o braço de controle
    # possa ser reexecutado NESTA sessão (mesma SEED, mesmo split, mesmo cache), em vez de
    # confrontarmos os resultados com números obtidos noutra execução.
    "dice_ce": (DiceCELoss, {"to_onehot_y": True, "softmax": True}),

    # PRINCIPAL — troca a Cross-Entropy por Focal, mantendo o termo Dice IDÊNTICO. A Focal
    # multiplica a CE por (1 - p)^gamma: voxels fáceis (fundo, miolo do fígado) quase somem do
    # gradiente, que se concentra nos difíceis (bordas e tumor). gamma=2.0 é o valor de
    # referência da literatura; lambda_* = 1.0 mantém os dois termos com o mesmo peso, como na
    # DiceCELoss — assim o único elemento alterado é a natureza do termo voxel-a-voxel.
    "dice_focal": (DiceFocalLoss, {"to_onehot_y": True, "softmax": True, "gamma": 2.0,
                                   "lambda_dice": 1.0, "lambda_focal": 1.0}),

    # ALTERNATIVA REGISTRADA — pondera cada classe por 1/volume². Mais agressiva no
    # desbalanceamento, mas instável aqui: com batch_size=1 e muitos volumes SEM tumor no GT, o
    # volume da classe rara no lote é zero e o peso degenera. Disponível e documentada; não é o
    # braço principal desta versão (ver §7.1).
    "generalized_dice": (GeneralizedDiceLoss, {"to_onehot_y": True, "softmax": True}),
}


def criar_perda(nome: str):
    """Fábrica de perdas — espelha criar_modelo() da Seção 6. Devolve (perda, kwargs usados)."""
    nome = nome.lower()
    if nome not in PERDAS_DISPONIVEIS:
        raise ValueError(
            f"Perda desconhecida: '{nome}'. Use uma de: {', '.join(PERDAS_DISPONIVEIS)}."
        )
    classe, kwargs = PERDAS_DISPONIVEIS[nome]
    return classe(**kwargs), dict(kwargs)


loss_fn, PERDA_PARAMS = criar_perda(PERDA)

# Métrica: INALTERADA em relação à v3.0 — não se troca a régua no mesmo experimento em que se
# troca o objeto medido. Dice POR CLASSE de primeiro plano; reduction="mean_batch" preserva a
# dimensão de classe -> vetor [fígado, tumor], em vez de um único escalar.
dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
CLASSES_FG = PROBLEMA.classes_fg    # ("figado", "tumor") — rótulos das colunas/linhas de log

# Pós-processamento para a métrica: previsão -> argmax + one-hot; rótulo -> one-hot.
post_pred = Compose([AsDiscrete(argmax=True, to_onehot=NUM_CLASSES)])
post_label = Compose([AsDiscrete(to_onehot=NUM_CLASSES)])

print(f"Perda selecionada : {PERDA}  ->  {loss_fn.__class__.__name__}")
for _k, _v in PERDA_PARAMS.items():
    print(f"{_k:>18} : {_v}")
print("Métrica (inalterada): Dice por classe", tuple(CLASSES_FG), "(include_background=False)")


## 7.2 · Otimizador

Usamos o **Adam**, um otimizador adaptativo robusto e amplamente empregado, com a
`learning_rate` e o `weight_decay` (regularização L2) definidos no `CONFIG`. Todo o protocolo de
otimização vem, portanto, da configuração central — nada é decidido aqui de forma implícita.

In [ ]:
usar_gpu = torch.cuda.is_available()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay,
)
print(f"Otimizador: Adam | lr={CONFIG.learning_rate} | weight_decay={CONFIG.weight_decay}")

## 7.3 · Diretório de resultados e registro do experimento

Cada experimento é uma **unidade preservada — nunca sobrescrita** (§8.3): salvamos o melhor
modelo, o histórico e o "prontuário" `REGISTRO_EXPERIMENTO` (ambiente + protocolo, das Seções 1–2).
Gravamos tudo no **Google Drive**, para que sobreviva ao encerramento da sessão do Colab.

> 🔑 **A chave da pasta mudou — e isso é uma correção necessária.** Na v3.0 o diretório era
> `resultados/<problema>/<arquitetura>/`. Como a v3.1 abre o eixo da **perda**, essa chave ficou
> **ambígua**: rodar `unet` + `dice_focal` gravaria **por cima** do resultado `unet` da v3.0, e o
> experimento anterior seria perdido silenciosamente — exatamente o que o §8.3 proíbe. A chave
> passa a ser o par **`<arquitetura>__<perda>`**.
>
> O sufixo é aplicado **sempre**, inclusive ao braço de controle `dice_ce`. É deliberado: as pastas
> da v3.0 se chamam `unet`, `unetr`, … (sem sufixo), então nenhuma execução da v3.1 pode colidir
> com elas. Os resultados da v3.0 continuam **intactos e legíveis** — a Seção 11 lê os dois
> formatos.
>
> 🛡️ **Guarda anti-sobrescrita.** Nomear bem não basta: um erro de digitação em `PERDA` ou
> `ARQUITETURA` ainda poderia apontar para uma pasta ocupada. Antes de gravar, comparamos a
> **assinatura** do experimento (`arquitetura`, `perda`, `problema`, `versao_notebook`) com a que
> já estiver no disco; se divergirem, a execução **para** — princípio de prevenção do §7.1:
> *qualquer inconsistência interrompe a execução antes de comprometer o experimento*. Reexecutar o
> **mesmo** experimento continua liberado.

In [ ]:
import os, json

# Diretório de resultados POR EXPERIMENTO. Na v3.0 a chave era só a arquitetura; como a v3.1 abre
# o eixo da PERDA, a chave passa a ser o PAR (arquitetura, perda) — do contrário esta execução
# gravaria por cima da pasta homônima da v3.0. O sufixo é aplicado SEMPRE, inclusive ao controle
# "dice_ce", para que as pastas legadas da v3.0 (sem sufixo) permaneçam intactas: cada experimento
# é uma unidade preservada, nunca sobrescrita (§8.3).
RESULTS_BASE = f"/content/drive/MyDrive/U-Net-Liver/resultados/{PROBLEMA.nome}"
EXPERIMENTO = f"{ARQUITETURA}__{PERDA}"
RESULTS_DIR = os.path.join(RESULTS_BASE, EXPERIMENTO)

# Assinatura do experimento: o conjunto de eixos que, se divergir, torna dois resultados coisas
# diferentes — e portanto proíbe que compartilhem o mesmo diretório.
ASSINATURA = {
    "arquitetura": ARQUITETURA,
    "perda": PERDA,
    "problema": PROBLEMA.nome,
    "versao_notebook": VERSAO_NOTEBOOK,
}

# Guarda anti-sobrescrita (§7.1: qualquer inconsistência interrompe a execução ANTES de
# comprometer o experimento). Bloqueia apenas divergência de EIXO — reexecutar o mesmo
# experimento (assinatura idêntica) continua permitido.
_resumo_previo = os.path.join(RESULTS_DIR, "resumo_experimento.json")
if os.path.isfile(_resumo_previo):
    with open(_resumo_previo, encoding="utf-8") as f:
        _antigo = json.load(f)
    _assinatura_antiga = {
        "arquitetura": _antigo.get("arquitetura"),
        "perda": _antigo.get("perda", "dice_ce"),   # v3.0 não registrava a perda: era DiceCELoss
        "problema": (_antigo.get("problema") or {}).get("nome"),
        "versao_notebook": (_antigo.get("ambiente") or {}).get("versao_notebook"),
    }
    if _assinatura_antiga != ASSINATURA:
        raise RuntimeError(
            f"Gravação abortada: já existe um experimento DIFERENTE em {RESULTS_DIR}\n"
            f"  no disco : {_assinatura_antiga}\n"
            f"  atual    : {ASSINATURA}\n"
            "Cada experimento é uma unidade preservada e nunca sobrescrita (DPC §8.3). "
            "Mova/renomeie a pasta antiga, ou revise ARQUITETURA (Seção 6) e PERDA (Seção 7.1)."
        )
    print("↻ Reexecução do MESMO experimento — assinatura confere, gravação liberada.")

os.makedirs(RESULTS_DIR, exist_ok=True)

# Prontuário do experimento: ambiente + protocolo (Seções 1-2) MAIS os eixos desta execução.
# Registrar a perda aqui é o que torna cada resultado autoexplicativo depois (§6.6).
REGISTRO_EXPERIMENTO_COMPLETO = {
    **REGISTRO_EXPERIMENTO,
    "arquitetura": ARQUITETURA,
    "perda": PERDA,
    "perda_params": PERDA_PARAMS,
}
with open(os.path.join(RESULTS_DIR, "registro_experimento.json"), "w", encoding="utf-8") as f:
    json.dump(REGISTRO_EXPERIMENTO_COMPLETO, f, indent=2, ensure_ascii=False)

print("Experimento :", EXPERIMENTO)
print("Resultados serão salvos em:", RESULTS_DIR)


## 7.4 · O laço de treino

Encapsulamos o treino numa **função** `treinar()` — código sem duplicação, com responsabilidade
clara (§7.2). Destaques metodológicos:

- **Precisão mista automática (AMP).** Em GPU, `autocast` + `GradScaler` reduzem memória e
  aceleram o treino sem perda prática de qualidade — importante nos limites do Colab. Em CPU, o
  recurso é automaticamente desativado.
- **Validação periódica.** A cada `val_interval` épocas, medimos o Dice no conjunto de
  validação (nunca no de teste, que permanece isolado — §6.1).
- **Salvamento do melhor modelo.** Sempre que o Dice de validação melhora, gravamos os pesos
  (`best_metric_model.pth`).
- **Parada antecipada (*early stopping*).** Se o Dice não melhora por `early_stopping_patience`
  validações, o treino para — evitando desperdício e sobreajuste (§6.1).
- **Histórico completo.** Perda de treino, perda e Dice de validação, tempo por época — salvos em
  `history.json` para auditoria e para as curvas de aprendizado.

In [ ]:
import time
from monai.data import decollate_batch

def treinar(model, train_loader, val_loader, loss_fn, optimizer, metric,
            max_epochs, val_interval, device, results_dir, patience, usar_gpu):
    scaler = torch.amp.GradScaler("cuda", enabled=usar_gpu)
    if usar_gpu:
        torch.cuda.reset_peak_memory_stats()

    historico = {"train_loss": [], "val_loss": [], "val_dice": [], "val_epochs": [], "epoch_time": []}
    melhor_dice, melhor_epoca, sem_melhora = -1.0, -1, 0

    for epoca in range(1, max_epochs + 1):
        t0 = time.time()

        # -------- Treino --------
        model.train()
        soma_loss = 0.0
        for lote in train_loader:
            inputs = lote["image"].to(device)
            labels = lote["label"].to(device)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=usar_gpu):
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            soma_loss += loss.item()
        loss_epoca = soma_loss / len(train_loader)
        historico["train_loss"].append(loss_epoca)
        linha = f"Época {epoca:>3}/{max_epochs} | perda treino: {loss_epoca:.4f}"

        # -------- Validação --------
        if epoca % val_interval == 0:
            model.eval()
            metric.reset()
            soma_val = 0.0
            with torch.no_grad():
                for vlote in val_loader:
                    v_in = vlote["image"].to(device)
                    v_lb = vlote["label"].to(device)
                    with torch.amp.autocast("cuda", enabled=usar_gpu):
                        v_out = model(v_in)
                        soma_val += loss_fn(v_out, v_lb).item()
                    preds = [post_pred(o) for o in decollate_batch(v_out)]
                    alvos = [post_label(o) for o in decollate_batch(v_lb)]
                    metric(y_pred=preds, y=alvos)
                dice_classes = metric.aggregate()              # tensor por classe: [fígado, tumor]
                val_dice = torch.nanmean(dice_classes).item()  # escalar de seleção = média foreground
                metric.reset()
            val_loss = soma_val / len(val_loader)
            historico["val_loss"].append(val_loss)
            historico["val_dice"].append(val_dice)
            historico.setdefault("val_dice_classes", []).append([float(x) for x in dice_classes])
            historico["val_epochs"].append(epoca)
            _det = "/".join(f"{c} {v:.3f}" for c, v in zip(CLASSES_FG, dice_classes.tolist()))
            linha += f" | perda val: {val_loss:.4f} | Dice [{_det}] média: {val_dice:.4f}"

            if val_dice > melhor_dice:
                melhor_dice, melhor_epoca, sem_melhora = val_dice, epoca, 0
                torch.save(model.state_dict(), os.path.join(results_dir, "best_metric_model.pth"))
                linha += "  ⭐"
            else:
                sem_melhora += 1

        historico["epoch_time"].append(time.time() - t0)
        print(linha + f" | {historico['epoch_time'][-1]:.1f}s")

        if sem_melhora >= patience:
            print(f"\n⏹️  Early stopping: {patience} validações sem melhora.")
            break

    # Persiste o histórico e reporta memória.
    with open(os.path.join(results_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(historico, f, indent=2)
    if usar_gpu:
        print(f"Memória GPU máxima: {torch.cuda.max_memory_allocated() / (1024 ** 3):.2f} GB")
    print(f"✅ Treino concluído. Melhor Dice de validação: {melhor_dice:.4f} (época {melhor_epoca}).")
    return historico, melhor_dice, melhor_epoca

print("Função treinar() definida.")

## 7.5 · Executar o treino

> ⏳ **Atenção ao tempo.** O protocolo completo prevê `CONFIG.max_epochs` (**600**) — o que pode
> levar **muitas horas** no Colab. Para **validar o laço**,
> comece com poucas épocas (`EPOCAS` abaixo) e confirme que a perda cai e o Dice sobe; depois,
> aumente para o experimento completo. Graças à parada antecipada, usar `CONFIG.max_epochs` como
> teto é seguro: o treino para sozinho quando a validação estagna.

In [ ]:
# Reancoragem da semente ANTES do treino: garante que a ordem de embaralhamento e o
# data augmentation sejam IDÊNTICOS entre as arquiteturas (a construção de modelos de
# tamanhos diferentes consome quantidades diferentes de aleatoriedade). Assim, a única
# diferença entre as execuções continua sendo a arquitetura (§9.2, comparação justa).
set_determinism(seed=CONFIG.seed)

# Épocas desta execução. Comece pequeno para validar; use CONFIG.max_epochs para o experimento completo.
EPOCAS = 5

historico, melhor_dice, melhor_epoca = treinar(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    metric=dice_metric,
    max_epochs=EPOCAS,                       # troque por CONFIG.max_epochs para o treino completo
    val_interval=CONFIG.val_interval,
    device=device,
    results_dir=RESULTS_DIR,
    patience=CONFIG.early_stopping_patience,
    usar_gpu=usar_gpu,
)

## 7.6 · Curvas de aprendizado

As curvas contam a história do treino: a **perda** deve cair e o **Dice de validação** deve subir.
Divergência entre treino e validação sinalizaria sobreajuste — algo que a parada antecipada ajuda
a conter.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.title("Perda por época")
plt.plot(range(1, len(historico["train_loss"]) + 1), historico["train_loss"], label="treino")
if historico["val_epochs"]:
    plt.plot(historico["val_epochs"], historico["val_loss"], marker="o", label="validação")
plt.xlabel("época"); plt.ylabel("perda"); plt.legend()

plt.subplot(1, 2, 2)
plt.title("Dice de validação")
plt.plot(historico["val_epochs"], historico["val_dice"], marker="o", color="green")
plt.xlabel("época"); plt.ylabel("Dice"); plt.ylim(0, 1)

plt.tight_layout(); plt.show()

## ✅ Resumo da Seção 7

O treinamento está implementado com componentes oficiais do MONAI e boas práticas de engenharia:

- **fábrica de perdas** `criar_perda()` com o seletor `PERDA` — `dice_ce` (controle),
  `dice_focal` (principal) e `generalized_dice` (alternativa) —, o **eixo isolado da v3.1**;
- **métrica** `DiceMetric` por classe, **inalterada** em relação à v3.0 (§6.5);
- **otimizador** Adam parametrizado pelo `CONFIG`, inalterado;
- laço `treinar()` com **AMP**, **validação periódica**, **salvamento do melhor modelo**,
  **early stopping** e **histórico** persistido no Drive (§6.4, §8.3);
- **curvas de aprendizado** para inspeção da convergência.

Os artefatos (`best_metric_model.pth`, `history.json`, `registro_experimento.json`) ficam em
`RESULTS_DIR` = `resultados/<problema>/<arquitetura>__<perda>/`, protegido pela **guarda
anti-sobrescrita** (§8.3). O melhor modelo alimentará a inferência.

> ⚙️ **Correções vs. referência (§8.2):** `softmax` no lugar de `sigmoid` para 2 classes;
> `DiceMetric` oficial no lugar de `1 - DiceLoss`; adição de AMP, early stopping e registro
> estruturado do experimento.

**➡️ Próxima seção — Inferência.** Carregaremos o melhor modelo e o aplicaremos a exames não
vistos com **janela deslizante** (`sliding_window_inference`), reconstruindo a máscara na
resolução de trabalho (§5.5).

# 8 · Inferência

Treinado o modelo, chega a hora de **aplicá-lo a exames que ele nunca viu** — o conjunto de
**teste**, mantido isolado desde a Seção 3 (§6.1). É a primeira vez que tocamos nesses dados, o
que preserva a validade da avaliação.

**Por que *sliding window* (§5.5)?** Modelos 3D são treinados em recortes de tamanho fixo
(`spatial_size`), mas exames reais podem ser maiores. A **inferência por janela deslizante**
(`sliding_window_inference`, componente oficial do MONAI) percorre o volume em janelas do tamanho
de treino e costura as previsões — com sobreposição entre janelas para suavizar as bordas. Mesmo
com nossos volumes já redimensionados, adotamos esse método porque é o padrão robusto e mantém o
pipeline pronto para inferência em resolução cheia no futuro.

Nesta seção nós vamos:

1. **Carregar** o melhor modelo salvo (`best_metric_model.pth`);
2. Definir uma **função de inferência** com janela deslizante;
3. **Aplicá-la** a um exame de teste e **visualizar** a predição frente à referência.

> 🧭 **Escopo.** Aqui produzimos e inspecionamos as máscaras previstas (avaliação *qualitativa*).
> As **métricas quantitativas** sistemáticas (Dice, Hausdorff, IoU) sobre todo o conjunto de teste
> ficam para a **Seção 9 — Avaliação**.

## 8.1 · Carregar o melhor modelo

Recarregamos os pesos de `best_metric_model.pth` (salvos no Drive durante o treino) na arquitetura
definida na Seção 6. Assim, a inferência usa **o melhor estado** encontrado na validação — e a
seção pode ser executada numa **nova sessão** sem retreinar: basta reexecutar as Seções 1–6 e
7.1–7.4 (que reconstroem `model` e definem `RESULTS_DIR`) e então esta célula.

In [ ]:
best_path = os.path.join(RESULTS_DIR, "best_metric_model.pth")
if not os.path.exists(best_path):
    raise FileNotFoundError(
        f"Modelo não encontrado em '{best_path}'. Execute o treino (Seção 7.5) "
        "ou confirme o RESULTS_DIR."
    )

model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
model.eval()
print("✅ Melhor modelo carregado de:", best_path)

## 8.2 · Função de inferência (janela deslizante)

Encapsulamos a inferência numa função. Passos:

1. `sliding_window_inference` percorre o volume em janelas de `roi_size` (= `spatial_size`), com
   **25% de sobreposição**, e devolve os *logits* (2 canais: fundo, fígado);
2. `softmax` converte os *logits* em probabilidades;
3. `argmax` escolhe, em cada voxel, a classe mais provável → a **máscara final** `{0, 1}`.

Usamos precisão mista (`autocast`) em GPU, como no treino, para eficiência.

In [ ]:
from monai.inferers import sliding_window_inference

@torch.no_grad()
def inferir(volume, roi_size=None, sw_batch_size=4, overlap=0.25):
    # volume: tensor (B, 1, H, W, D). Retorna (mascara {0,1}, probabilidades).
    roi_size = tuple(roi_size or CONFIG.spatial_size)
    model.eval()
    with torch.amp.autocast("cuda", enabled=usar_gpu):
        logits = sliding_window_inference(
            volume.to(device), roi_size, sw_batch_size, model, overlap=overlap,
        )
    probs = torch.softmax(logits.float(), dim=1)
    mascara = torch.argmax(probs, dim=1, keepdim=True)   # (B, 1, H, W, D)
    return mascara, probs

print("Função inferir() definida (roi_size =", tuple(CONFIG.spatial_size), ").")

## 8.3 · Inferência num exame de teste

Aplicamos o modelo ao **primeiro exame do conjunto de teste** e comparamos, em alguns cortes
axiais que contêm fígado, a **referência** (verde) com a **predição** (vermelho). Como contexto
rápido, calculamos também o Dice deste exame com a métrica **oficial** `DiceMetric` — a avaliação
completa vem na Seção 9.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from monai.data import decollate_batch

# Um exame de teste (test_transforms já aplica o mesmo pré-processamento determinístico).
exame = test_ds[0]
volume = exame["image"].unsqueeze(0)          # (1, 1, H, W, D)
# A predição sai da inferência na GPU; movemos a referência para o mesmo dispositivo,
# pois a DiceMetric exige y_pred e y no mesmo device.
referencia = exame["label"].to(device)         # (1, H, W, D)

mascara, _ = inferir(volume)

# Dice deste exame (métrica oficial, apenas o fígado).
dice_1 = DiceMetric(include_background=False, reduction="mean")
pred_oh = [AsDiscrete(to_onehot=NUM_CLASSES)(p) for p in decollate_batch(mascara)]
ref_oh = [AsDiscrete(to_onehot=NUM_CLASSES)(g) for g in decollate_batch(referencia.unsqueeze(0))]
dice_1(y_pred=pred_oh, y=ref_oh)
print(f"Dice (fígado) deste exame: {dice_1.aggregate().item():.4f}")

img_np = volume[0, 0].cpu().numpy()
gt_np = referencia[0].cpu().numpy()
pred_np = mascara[0, 0].cpu().numpy()

# Escolhe 3 cortes que contêm fígado na referência.
z_figado = np.where(gt_np.sum(axis=(0, 1)) > 0)[0]
zs = (np.linspace(z_figado.min(), z_figado.max(), 3).astype(int)
      if len(z_figado) else [img_np.shape[2] // 2])

plt.figure(figsize=(11, 3.6 * len(zs)))
for r, z in enumerate(zs):
    plt.subplot(len(zs), 3, r * 3 + 1); plt.title(f"CT (z={z})")
    plt.imshow(img_np[:, :, z], cmap="gray"); plt.axis("off")
    plt.subplot(len(zs), 3, r * 3 + 2); plt.title("Referência")
    plt.imshow(img_np[:, :, z], cmap="gray")
    plt.imshow(np.ma.masked_where(gt_np[:, :, z] == 0, gt_np[:, :, z]), alpha=0.5, cmap="Greens", vmin=0, vmax=1); plt.axis("off")
    plt.subplot(len(zs), 3, r * 3 + 3); plt.title("Predição")
    plt.imshow(img_np[:, :, z], cmap="gray")
    plt.imshow(np.ma.masked_where(pred_np[:, :, z] == 0, pred_np[:, :, z]), alpha=0.5, cmap="Reds", vmin=0, vmax=1); plt.axis("off")
plt.tight_layout(); plt.show()

## ✅ Resumo da Seção 8

A inferência está operacional:

- **carregamos** o melhor modelo (`best_metric_model.pth`) na arquitetura da Seção 6;
- definimos `inferir()` com **`sliding_window_inference`** (componente oficial), *softmax* e *argmax*
  para produzir a máscara `{0, 1}`;
- **aplicamos** ao conjunto de teste isolado e **comparamos** predição × referência, com o Dice do
  exame como contexto.

A função `inferir()` será reutilizada na avaliação.

**➡️ Próxima seção — Avaliação.** Percorreremos **todo o conjunto de teste** calculando métricas
quantitativas — **Dice**, **Hausdorff** e **IoU** (§6.5) —, com estatísticas resumidas e
inspeção dos melhores e piores casos, unindo avaliação quantitativa e qualitativa.

# 9 · Avaliação Quantitativa e Qualitativa

Um modelo só pode ser julgado por como se sai em dados que **nunca viu**. Aqui percorremos **todo
o conjunto de teste** e medimos o desempenho — mas o DPC (§6.5) é explícito: *nenhuma conclusão se
baseia em uma única métrica*. Combinamos três medidas complementares e, além dos números, uma
**inspeção visual** dos casos extremos, para revelar falhas sistemáticas que médias globais
escondem.

**As três métricas (todas sobre o fígado, sem o fundo):**

- **Dice** — sobreposição entre predição e referência (0 a 1, ↑ melhor). A métrica principal.
- **Hausdorff 95% (HD95)** — a maior distância entre as bordas das duas máscaras, no percentil 95
  (**em milímetros**, ↓ melhor). Captura erros de **contorno** que o Dice, focado em volume, pode
  não penalizar. O percentil 95 a torna robusta a *outliers* isolados.
- **IoU (Jaccard)** — outra medida de sobreposição (0 a 1, ↑ melhor), mais severa que o Dice.

Nesta seção nós vamos:

1. Definir as **três métricas** (componentes oficiais do MONAI);
2. **Avaliar todo o conjunto de teste**, com estatísticas resumidas e persistência em CSV;
3. Visualizar a **distribuição** das métricas;
4. Inspecionar **o melhor e o pior caso** (avaliação qualitativa).

## 9.1 · As métricas

Usamos as métricas oficiais do MONAI (§5.4), todas com `include_background=False` — o Dice sobre o
fundo (enorme) mascararia o desempenho no órgão. As máscaras são convertidas para *one-hot* antes
da medição, como o MONAI espera.

> 📏 **HD95 em milímetros (§8.2).** O `Resized` deixa cada exame com um tamanho de voxel físico
> diferente (a mesma grade 128×128×64 abriga fígados de extensões distintas). Se medíssemos a
> Hausdorff em voxels, estaríamos **misturando unidades** entre pacientes. Por isso o HD95 é
> calculado por exame informando o **espaçamento físico real** daquele volume (extraído do `affine`
> da grade redimensionada) — assim o resultado sai em **mm** e é comparável entre pacientes. Dice e
> IoU, por serem razões de sobreposição, não sofrem esse efeito.

In [ ]:
from monai.metrics import DiceMetric, MeanIoU, compute_hausdorff_distance
from monai.data.utils import affine_to_spacing
from monai.transforms import AsDiscrete

dice_metric_eval = DiceMetric(include_background=False, reduction="mean")
iou_metric = MeanIoU(include_background=False, reduction="mean")
# O HD95 é calculado por exame com a função compute_hausdorff_distance, para podermos
# informar o espaçamento físico (mm) específico de cada volume — ver célula seguinte.

para_onehot = AsDiscrete(to_onehot=NUM_CLASSES)
print("Métricas prontas: Dice, HD95 (mm) e IoU — reportadas POR CLASSE (fígado, tumor).")

## 9.2 · Avaliação sobre todo o conjunto de teste

Percorremos os exames de teste, aplicamos a inferência (reutilizando `inferir()` da Seção 8) e
registramos as três métricas por exame. O resultado é consolidado numa tabela, resumido em
estatísticas (média, desvio, mínimo, mediana, máximo) e **salvo em CSV** — cada experimento é uma
unidade preservada (§8.3).

> ℹ️ O **HD95** pode ficar indefinido (infinito) se, para algum exame, a predição não encontrar
> fígado algum. Tratamos esses casos como ausentes (`NaN`) no resumo, e eles aparecem como Dice
> baixo — o que a inspeção qualitativa ajuda a entender.
>
> ⚙️ **Nota técnica (§8.2).** O `HD95` é calculado na **CPU**: na GPU, o backend de distâncias de
> superfície do MONAI exige uma compilação CUDA/Thrust (C++17) que falha no runtime atual do
> Colab. Dice e IoU seguem na GPU; só o HD95 tem seus tensores movidos para a CPU.

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from monai.data import decollate_batch

registros = []
for idx in tqdm(range(len(test_ds)), desc="Avaliando conjunto de teste"):
    exame = test_ds[idx]
    volume = exame["image"].unsqueeze(0)
    referencia = exame["label"].to(device)        # mesmo dispositivo da predição
    mascara, _ = inferir(volume)

    pred_oh = [para_onehot(p) for p in decollate_batch(mascara)]
    ref_oh = [para_onehot(g) for g in decollate_batch(referencia.unsqueeze(0))]

    # Métricas por classe de primeiro plano: vetor [fígado, tumor]. Tumor ausente no GT -> NaN
    # (ignore_empty), agregado adiante só sobre os exames que contêm tumor.
    d = dice_metric_eval(pred_oh, ref_oh)[0]
    j = iou_metric(pred_oh, ref_oh)[0]

    # HD95 em MILÍMETROS: informamos o espaçamento físico real desta grade redimensionada
    # (extraído do affine), para que a distância seja comparável entre pacientes.
    # Calculado na CPU: na GPU, o backend de distâncias de superfície do MONAI dispara uma
    # compilação CUDA/Thrust (C++17) que falha no runtime atual do Colab.
    espacamento = affine_to_spacing(referencia.affine).tolist()
    pred_b = torch.stack(pred_oh).cpu()   # (1, C, H, W, D)
    ref_b = torch.stack(ref_oh).cpu()
    h = compute_hausdorff_distance(
        pred_b, ref_b, include_background=False, percentile=95, spacing=espacamento,
    )[0]

    dice_metric_eval.reset(); iou_metric.reset()

    registros.append({
        "exame": os.path.basename(test_files[idx]["image"]),
        "dice_figado": d[0].item(), "dice_tumor": d[1].item(),
        "iou_figado": j[0].item(),  "iou_tumor": j[1].item(),
        "hd95_figado": h[0].item(), "hd95_tumor": h[1].item(),
    })

df_metricas = pd.DataFrame(registros)
csv_path = os.path.join(RESULTS_DIR, "metricas_teste.csv")
df_metricas.to_csv(csv_path, index=False)

# Resumo por classe (HD95 infinito -> NaN; estatísticas de tumor só onde há tumor no GT).
resumo = df_metricas.copy()
for c in ["hd95_figado", "hd95_tumor"]:
    resumo[c] = resumo[c].replace([np.inf, -np.inf], np.nan)
cols = ["dice_figado", "dice_tumor", "iou_figado", "iou_tumor", "hd95_figado", "hd95_tumor"]
n_tumor = int(df_metricas["dice_tumor"].notna().sum())
print(f"Resumo sobre {len(df_metricas)} exames de teste ({n_tumor} contêm tumor no GT):\n")
print(resumo[cols].describe().loc[["mean", "std", "min", "50%", "max"]].round(4))
print("\nMétricas por exame salvas em:", csv_path)

## 9.3 · Distribuição das métricas

Médias não contam a história toda: um bom desempenho médio pode esconder poucos exames muito
ruins. Os *boxplots* mostram a **dispersão** de cada métrica entre os exames — a caixa cobre a
metade central, e pontos afastados sinalizam casos atípicos que merecem atenção.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

dados = [df_metricas["dice_figado"].dropna(), df_metricas["dice_tumor"].dropna(),
         df_metricas["iou_figado"].dropna(), df_metricas["iou_tumor"].dropna()]
ax[0].boxplot(dados)
ax[0].set_xticks([1, 2, 3, 4]); ax[0].set_xticklabels(["Dice fíg", "Dice tum", "IoU fíg", "IoU tum"])
ax[0].set_title("Dice e IoU por classe  (↑ melhor)"); ax[0].set_ylim(0, 1); ax[0].grid(axis="y", alpha=0.3)

hf = df_metricas["hd95_figado"].replace([np.inf, -np.inf], np.nan).dropna()
ht = df_metricas["hd95_tumor"].replace([np.inf, -np.inf], np.nan).dropna()
ax[1].boxplot([hf, ht])
ax[1].set_xticks([1, 2]); ax[1].set_xticklabels(["HD95 fíg", "HD95 tum"])
ax[1].set_title("Hausdorff 95%  (↓ melhor, em mm)"); ax[1].grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.show()

## 9.4 · Inspeção do melhor e do pior caso

Aqui a avaliação quantitativa encontra a qualitativa (§6.5). Olhar o **pior caso** costuma ser o
mais informativo: revela *como* o modelo falha — bordas imprecisas, um lobo hepático perdido,
confusão com órgãos vizinhos de baixo contraste — falhas sistemáticas que uma média jamais
mostraria. O **melhor caso** serve de referência do que o modelo já faz bem.

In [ ]:
# Seleção do melhor/pior caso pela classe DIFÍCIL (tumor), que é a mais informativa para
# a inspeção visual. Se nenhum exame de teste tiver tumor, recai sobre o fígado.
coluna_sel = "dice_tumor" if df_metricas["dice_tumor"].notna().any() else "dice_figado"
validos = df_metricas.dropna(subset=[coluna_sel])
melhor = validos.loc[validos[coluna_sel].idxmax()]
pior = validos.loc[validos[coluna_sel].idxmin()]
print(f"Critério de seleção: {coluna_sel}")
print(f"Melhor caso: {melhor['exame']}  —  {coluna_sel} {melhor[coluna_sel]:.4f}")
print(f"Pior caso  : {pior['exame']}  —  {coluna_sel} {pior[coluna_sel]:.4f}")


def visualizar_caso(idx, titulo):
    exame = test_ds[idx]
    volume = exame["image"].unsqueeze(0)
    referencia = exame["label"].to(device)
    mascara, _ = inferir(volume)

    img = volume[0, 0].cpu().numpy()
    g = referencia[0].cpu().numpy()
    p = mascara[0, 0].cpu().numpy()

    # Cortes onde há tumor (2) se existir; senão, onde há fígado.
    alvo = (g == 2) if (g == 2).any() else (g > 0)
    z_alvo = np.where(alvo.sum(axis=(0, 1)) > 0)[0]
    zs = (np.linspace(z_alvo.min(), z_alvo.max(), 3).astype(int)
          if len(z_alvo) else [img.shape[2] // 2])

    # Overlay multiclasse: fígado (1) e tumor (2) em cores distintas (vmin=1, vmax=2).
    plt.figure(figsize=(11, 3.6 * len(zs)))
    for r, z in enumerate(zs):
        plt.subplot(len(zs), 3, r * 3 + 1); plt.title(f"CT (z={z})")
        plt.imshow(img[:, :, z], cmap="gray"); plt.axis("off")
        plt.subplot(len(zs), 3, r * 3 + 2); plt.title("Referência (fíg/tum)")
        plt.imshow(img[:, :, z], cmap="gray")
        plt.imshow(np.ma.masked_where(g[:, :, z] == 0, g[:, :, z]),
                   alpha=0.5, cmap="autumn", vmin=1, vmax=2); plt.axis("off")
        plt.subplot(len(zs), 3, r * 3 + 3); plt.title("Predição (fíg/tum)")
        plt.imshow(img[:, :, z], cmap="gray")
        plt.imshow(np.ma.masked_where(p[:, :, z] == 0, p[:, :, z]),
                   alpha=0.5, cmap="autumn", vmin=1, vmax=2); plt.axis("off")
    plt.suptitle(titulo, y=1.002, fontsize=13)
    plt.tight_layout(); plt.show()


visualizar_caso(int(pior.name), f"PIOR caso — {pior['exame']} ({coluna_sel} {pior[coluna_sel]:.4f})")
visualizar_caso(int(melhor.name), f"MELHOR caso — {melhor['exame']} ({coluna_sel} {melhor[coluna_sel]:.4f})")

## ✅ Resumo da Seção 9

A avaliação da *baseline* está completa, quantitativa e qualitativamente:

- medimos **Dice, HD95 e IoU** sobre **todo o conjunto de teste**, com componentes oficiais do MONAI (§6.5);
- consolidamos os resultados por exame numa tabela **salva em CSV** (`metricas_teste.csv`, §8.3);
- analisamos a **distribuição** das métricas (dispersão, casos atípicos);
- inspecionamos **o pior e o melhor caso**, unindo número e imagem para revelar falhas sistemáticas.

Esses resultados caracterizam a baseline v1.0 e servirão de **referência** para comparar as
arquiteturas das versões futuras, sob exatamente o mesmo protocolo (§9.2).

**➡️ Próxima seção — Visualização e Persistência.** Aprofundaremos a visualização (sobreposições,
múltiplos cortes) e organizaremos a **persistência dos artefatos** do experimento — pesos,
métricas, histórico e configuração —, consolidando o registro reproduzível (§5.5, §8.3).

# 10 · Visualização e Persistência

Esta seção fecha o ciclo experimental com duas tarefas (§5.5):

- **Visualização** — sobreposições em múltiplos cortes e uma **reconstrução 3D** do fígado
  previsto, que comunicam o resultado de forma mais rica que uma métrica isolada;
- **Persistência** — consolidar e registrar todos os **artefatos** do experimento (pesos,
  métricas, histórico, configuração), para que ele seja uma **unidade preservada e reproduzível**
  (§8.3).

Nesta seção nós vamos:

1. Gerar uma **montagem de cortes** com a sobreposição referência × predição;
2. Produzir uma **reconstrução 3D** da máscara prevista;
3. **Consolidar os artefatos** num resumo do experimento e listar tudo o que ficou salvo.

## 10.1 · Montagem de cortes

Uma visão em vários cortes axiais dá uma noção do desempenho ao longo de **todo o volume**, não
só numa fatia. Escolhemos um caso **representativo** (Dice próximo da mediana do teste) e
sobrepomos, em cada corte, a **referência (verde)** e a **predição (vermelho)** — a concordância
aparece em tons alaranjados (verde + vermelho).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Caso representativo: Dice mais próximo da mediana do conjunto de teste.
# A variável 'coluna_sel' é definida na célula anterior para usar a métrica mais relevante.
idx_repr = int((df_metricas[coluna_sel] - df_metricas[coluna_sel].median()).abs().idxmin())
exame = test_ds[idx_repr]
volume = exame["image"].unsqueeze(0)
referencia = exame["label"].to(device)
mascara, _ = inferir(volume)

img = volume[0, 0].cpu().numpy()
gt = referencia[0].cpu().numpy()
pred = mascara[0, 0].cpu().numpy()

# 12 cortes igualmente espaçados na faixa que contém fígado.
z_figado = np.where(gt.sum(axis=(0, 1)) > 0)[0]
zs = np.linspace(z_figado.min(), z_figado.max(), 12).astype(int)

plt.figure(figsize=(12, 9))
for k, z in enumerate(zs):
    plt.subplot(3, 4, k + 1)
    plt.imshow(img[:, :, z], cmap="gray")
    plt.imshow(np.ma.masked_where(gt[:, :, z] == 0, gt[:, :, z]), cmap="Greens", alpha=0.45, vmin=0, vmax=1)
    plt.imshow(np.ma.masked_where(pred[:, :, z] == 0, pred[:, :, z]), cmap="Reds", alpha=0.45, vmin=0, vmax=1)
    plt.title(f"z={z}", fontsize=8); plt.axis("off")
# Usar 'coluna_sel' para o título também, formatando para melhor legibilidade.
plt.suptitle(f"{df_metricas.loc[idx_repr, 'exame']} — verde: referência · vermelho: predição "
             f"({coluna_sel.replace('_', ' ')} {df_metricas.loc[idx_repr, coluna_sel]:.3f})", fontsize=12)
plt.tight_layout(); plt.show()

## 10.2 · Reconstrução 3D do fígado previsto

A partir da máscara volumétrica, o algoritmo ***marching cubes*** (do `scikit-image`) extrai a
**superfície** do órgão, que renderizamos em 3D. Essa visão tridimensional aproxima o resultado do
uso clínico (volumetria, planejamento cirúrgico, §2.2) e ajuda a perceber a forma global da
segmentação — algo que cortes 2D isolados não transmitem.

In [ ]:
# Reconstrói a superfície da máscara prevista (mesmo caso da montagem acima).
try:
    from skimage.measure import marching_cubes
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection

    # step_size > 1 reduz o número de faces (renderização mais leve).
    verts, faces, _, _ = marching_cubes(pred.astype(float), level=0.5, step_size=2)

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection="3d")
    malha = Poly3DCollection(verts[faces], alpha=0.6)
    malha.set_facecolor((0.80, 0.25, 0.25))
    ax.add_collection3d(malha)
    ax.set_xlim(0, pred.shape[0]); ax.set_ylim(0, pred.shape[1]); ax.set_zlim(0, pred.shape[2])
    ax.set_title("Reconstrução 3D do fígado previsto")
    ax.set_axis_off()
    plt.tight_layout(); plt.show()
except Exception as erro:
    print("Não foi possível gerar a reconstrução 3D:", erro)

## 10.3 · Consolidação e persistência dos artefatos

Por fim, reunimos o **resumo do experimento** — ambiente, configuração e o desempenho no teste —
num único arquivo (`resumo_experimento.json`) e listamos tudo o que ficou salvo em `RESULTS_DIR`.
Assim, o experimento fica **autocontido e auditável** (§6.6, §8.3): qualquer pessoa consegue saber
*em que ambiente*, *com qual protocolo* e *com que resultado* ele foi produzido.

In [ ]:
import os, json

# Resumo das métricas de teste — POR CLASSE (HD95 infinito -> NaN; médias ignoram NaN, então
# as estatísticas de tumor consideram apenas os exames que contêm tumor no GT).
resumo_metricas = {"n_exames_teste": int(len(df_metricas))}
for cls in PROBLEMA.classes_fg:
    dser = df_metricas[f"dice_{cls}"]
    iser = df_metricas[f"iou_{cls}"]
    hser = df_metricas[f"hd95_{cls}"].replace([np.inf, -np.inf], np.nan)
    resumo_metricas[f"dice_{cls}_media"] = float(dser.mean())
    resumo_metricas[f"dice_{cls}_desvio"] = float(dser.std())
    resumo_metricas[f"iou_{cls}_media"] = float(iser.mean())
    resumo_metricas[f"hd95_{cls}_mediana"] = float(hser.median())
    resumo_metricas[f"n_exames_{cls}"] = int(dser.notna().sum())

# Inclui o melhor Dice de validação (média foreground), se o treino foi executado nesta sessão.
try:
    resumo_metricas["melhor_dice_val"] = float(melhor_dice)
    resumo_metricas["melhor_epoca"] = int(melhor_epoca)
except NameError:
    pass

# Metadados da arquitetura desta execução — usados na comparação (Seção 11).
try:
    resumo_metricas["n_params"] = int(n_params)
except NameError:
    pass

resumo_experimento = {
    "experimento": EXPERIMENTO,           # chave da pasta: "<arquitetura>__<perda>"
    "arquitetura": ARQUITETURA,           # Seção 6 (fixa nesta versão)
    "perda": PERDA,                       # Seção 7.1 (A variável comparada na v3.1)
    "perda_params": PERDA_PARAMS,         # hiperparâmetros efetivos da perda (§6.6)
    "problema": asdict(PROBLEMA),         # Seção 2 (define a ilha de comparabilidade)
    "ambiente": AMBIENTE,                 # Seção 1
    "configuracao": asdict(CONFIG),       # Seção 2
    "metricas_teste": resumo_metricas,    # Seção 9
}
with open(os.path.join(RESULTS_DIR, "resumo_experimento.json"), "w", encoding="utf-8") as f:
    json.dump(resumo_experimento, f, indent=2, ensure_ascii=False)

print("Resumo do experimento:")
print(json.dumps(resumo_metricas, indent=2, ensure_ascii=False))

print("\nArtefatos salvos em", RESULTS_DIR)
print("-" * 52)
for nome in sorted(os.listdir(RESULTS_DIR)):
    tam_kb = os.path.getsize(os.path.join(RESULTS_DIR, nome)) / 1024
    print(f"  {nome:34s} {tam_kb:9.1f} KB")

## ✅ Resumo da Seção 10

O ciclo experimental está encerrado e preservado:

- **visualizamos** o resultado em uma montagem de múltiplos cortes (referência × predição);
- geramos uma **reconstrução 3D** da máscara prevista com *marching cubes*;
- **consolidamos** o experimento em `resumo_experimento.json` e listamos todos os artefatos
  salvos no Drive (pesos, histórico, métricas por exame, configuração e resumo).

O experimento é agora uma **unidade autocontida e reproduzível** (§8.3): ambiente, protocolo,
modelo e resultados podem ser auditados e reconstruídos.

**➡️ Próxima seção — Comparação entre Arquiteturas.** Depois de rodar o notebook uma vez para cada
arquitetura, agregaremos os resultados salvos e compararemos U-Net, SegResNet e DynUNet sob o
protocolo idêntico (§9.2).

# 11 · Comparação entre Experimentos (arquitetura × perda)

Esta seção agrega e compara os experimentos **no problema multiclasse**. Na v3.0 um experimento era
identificado apenas pela **arquitetura**; a partir da v3.1 ele é o par **(arquitetura, perda)** —
por isso a seção lê `resultados/figado_multiclasse/<arquitetura>__<perda>/` e reporta a perda como
uma coluna própria, **por classe** (fígado e tumor).

> 🔁 **Compatível com a v3.0.** As pastas antigas (`unet`, `unetr`, …, sem sufixo) continuam sendo
> lidas: seus `resumo_experimento.json` não têm o campo `perda`, e o carregador assume `dice_ce` —
> que é, de fato, a perda que a v3.0 usava. Assim os resultados anteriores **entram na mesma
> tabela**, sem serem alterados nem reprocessados.

> ⚠️ **Ilha de comparabilidade.** A **tarefa** não mudou entre a v3.0 e a v3.1 — mesmo `Problema`,
> mesmo pré-processamento, mesmo `CONFIG`. Logo, estes números **são** comparáveis aos da v3.0, e
> **não** aos do fígado binário (v1.0–v2.0). Como só a perda variou, a diferença entre linhas com a
> mesma arquitetura é atribuível **à perda**.

> ▶️ **Pré-requisito.** Rode o notebook inteiro **uma vez para cada `PERDA`** que quiser comparar
> (mantendo `ARQUITETURA` fixa). A tabela mostra apenas os experimentos com resultados no Drive.
>
> 🔌 **Roda isolada.** Esta seção é **autossuficiente**: monta o Drive e localiza os resultados por
> conta própria — não é preciso rodar as Seções 1–10 de novo só para ver a comparação.

Nesta seção nós vamos:

1. **Carregar** os resumos de todos os experimentos executados (v3.0 e v3.1);
2. **Tabular** as métricas de teste **por classe** (Dice fígado/tumor, HD95, IoU) e o custo;
3. **Isolar o eixo da v3.1** numa tabela (arquitetura × perda) do Dice de tumor;
4. **Comparar visualmente** o desempenho por experimento.

## 11.1 · Carregar os resultados salvos

Varremos as subpastas de `resultados/figado_multiclasse/`, lendo o `resumo_experimento.json` de
cada arquitetura executada. O resultado é uma tabela comparativa **por classe** — a evidência
central da v3.0. Ordenamos pelo **Dice de tumor**, a classe mais difícil.

In [ ]:
import os, json, glob
import pandas as pd

# Esta seção é AUTOSSUFICIENTE: pode ser executada isoladamente (ex.: num dia em que você não
# treina nada, só quer regerar a comparação). Se RESULTS_BASE ainda não foi definido nesta sessão
# (porque você não rodou a Seção 7), montamos o Drive e assumimos o caminho padrão dos resultados.
if "RESULTS_BASE" not in globals():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount("/content/drive")          # idempotente: se já montado, apenas confirma
    except ImportError:
        pass
    RESULTS_BASE = "/content/drive/MyDrive/U-Net-Liver/resultados/figado_multiclasse"

print("Lendo resultados de:", RESULTS_BASE)

linhas = []
for pasta in sorted(glob.glob(os.path.join(RESULTS_BASE, "*"))):
    caminho = os.path.join(pasta, "resumo_experimento.json")
    if not os.path.isfile(caminho):
        continue
    with open(caminho, encoding="utf-8") as f:
        r = json.load(f)
    m = r.get("metricas_teste", {})
    nome_pasta = os.path.basename(pasta)

    # Compatibilidade com a v3.0: lá as pastas se chamavam "<arquitetura>" (sem sufixo) e o
    # resumo não registrava a perda — que era sempre DiceCELoss. Os defaults abaixo recuperam
    # essa informação sem tocar nos arquivos antigos.
    linhas.append({
        "experimento": nome_pasta,
        "arquitetura": r.get("arquitetura", nome_pasta.split("__")[0]),
        "perda": r.get("perda", "dice_ce"),
        "versao": (r.get("ambiente") or {}).get("versao_notebook", "?"),
        "dice_figado": m.get("dice_figado_media"),
        "dice_tumor": m.get("dice_tumor_media"),
        "hd95_tumor_mm": m.get("hd95_tumor_mediana"),
        "iou_tumor": m.get("iou_tumor_media"),
        "melhor_epoca": m.get("melhor_epoca"),
        "n_params": m.get("n_params"),
    })

if not linhas:
    print("Nenhum resultado encontrado em", RESULTS_BASE,
          "\nRode o notebook uma vez para cada PERDA antes desta seção.")
else:
    df_comp = (pd.DataFrame(linhas)
                 .sort_values("dice_tumor", ascending=False, na_position="last")
                 .reset_index(drop=True))
    print("Experimentos — problema multiclasse (ordenado por Dice de tumor):\n")
    print(df_comp.round(4).to_string(index=False))

    # Leitura do EIXO da v3.1: para cada arquitetura, o efeito de trocar a perda. Só faz sentido
    # (e só é impresso) quando há mais de uma perda no disco.
    #
    # A VERSÃO entra no índice de propósito: o controle "dice_ce" existe duas vezes no disco —
    # a execução original da v3.0 e a reexecução pareada da v3.1. São sessões distintas, e
    # fundi-las numa média esconderia justamente a informação que interessa. Mantendo-as em
    # linhas separadas, dá para ler as duas coisas: o efeito da perda (dentro da linha 3.1) e a
    # reprodutibilidade entre sessões (comparando as duas linhas na coluna dice_ce).
    if df_comp["perda"].nunique() > 1:
        pivo = df_comp.pivot_table(index=["arquitetura", "versao"], columns="perda",
                                   values="dice_tumor", aggfunc="mean")
        print("\nEixo isolado da v3.1 — Dice de TUMOR por (arquitetura x versão x perda):\n")
        print(pivo.round(4).to_string())
    else:
        print("\n(Apenas uma perda no disco — rode outra PERDA para ver o eixo da v3.1.)")


## 11.2 · Comparação visual

Um gráfico deixa a comparação imediata. Mostramos o **Dice médio por classe** (↑ melhor) e o
**HD95 mediano do tumor em mm** (↓ melhor) por **experimento** — o rótulo do eixo x é
`<arquitetura>__<perda>` (ou só `<arquitetura>`, nos resultados legados da v3.0).

Duas leituras importam aqui, e a segunda é a que a v3.1 foi construída para permitir:

- **entre arquiteturas** com a mesma perda — a leitura herdada da v3.0. O DPC (§10.4) pede
  considerar **desempenho *e* custo**: a coluna `n_params` informa o tamanho de cada rede.
- **entre perdas** com a mesma arquitetura — **a leitura da v3.1**. Como tudo o mais foi mantido
  constante, qualquer diferença na barra laranja (tumor) é atribuível à função de perda. Observe
  também a barra azul (fígado): uma perda que salva o tumor **às custas** do fígado não é um ganho,
  é uma troca — e precisa ser reportada como tal.

In [ ]:
import matplotlib.pyplot as plt

if linhas:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    x = list(range(len(df_comp))); w = 0.38
    rotulos = df_comp["experimento"]     # "<arquitetura>__<perda>" (ou legado da v3.0)

    ax[0].bar([i - w / 2 for i in x], df_comp["dice_figado"], width=w, label="fígado", color="#4C78A8")
    ax[0].bar([i + w / 2 for i in x], df_comp["dice_tumor"], width=w, label="tumor", color="#F58518")
    ax[0].set_xticks(x); ax[0].set_xticklabels(rotulos, rotation=20, ha="right")
    ax[0].set_title("Dice médio por classe  (↑ melhor)")
    ax[0].set_ylim(0, 1); ax[0].legend(); ax[0].grid(axis="y", alpha=0.3)

    ax[1].bar(rotulos, df_comp["hd95_tumor_mm"], color="#E45756")
    ax[1].tick_params(axis="x", labelrotation=20)
    ax[1].set_title("HD95 do tumor, mediano  (↓ melhor, em mm)")
    ax[1].grid(axis="y", alpha=0.3)

    plt.tight_layout(); plt.show()
else:
    print("Sem dados para plotar — execute o notebook para pelo menos uma PERDA primeiro.")


## ✅ Resumo da Seção 11

Comparamos os experimentos **no problema multiclasse**, sob protocolo **idêntico**, reportando Dice
**por classe** e identificando cada resultado pelo par **(arquitetura, perda)**. O fígado tende a
atingir Dice alto; o **tumor** é a classe difícil — pequeno, de contraste variável e fortemente
desbalanceado —, e é nele que o eixo da v3.1 se manifesta, se é que se manifesta.

**Como ler a tabela cruzada.** Linhas = arquitetura, colunas = perda, células = Dice de tumor.
Como *tudo* o mais foi mantido constante entre as colunas, a variação horizontal é atribuível à
**função de perda** — é essa a evidência que a v3.1 produz.

> ⚠️ **Um resultado nulo aqui também é resultado.** Se `dice_focal` não afastar o tumor de zero, a
> conclusão registrada é que a cegueira ao desbalanceamento **não era a causa dominante** — o que
> aponta o dedo para o **redimensionamento global** (`spatial_size`) e transforma a v3.2 numa
> hipótese com evidência, e não num palpite. Relatar isso é obrigação, não fracasso (§5.1,
> *Transparência*).

> ⚠️ Lembrete: esta é a ilha de comparabilidade **multiclasse** — não confronte estes números com
> os do fígado binário (v1.0–v2.0). E o protocolo continua **sem afinação por classe**: o que
> mudou foi um elemento declarado do protocolo, não um ajuste oportunista.

Cada experimento permanece preservado em `resultados/figado_multiclasse/<arquitetura>__<perda>/`,
e os da v3.0, em suas pastas originais — nenhum foi sobrescrito (§8.3).

**➡️ Próxima seção — Conclusões.** Fechamento científico da v3.1.

# 12 · Conclusões

Concluímos a **v3.0**: a plataforma deixou de tratar o fígado como órgão único e passou à
**segmentação multiclasse** (fígado + tumor), abrindo o primeiro **eixo de problema** além da
arquitetura. Fez isso **nascendo parametrizada** — via o objeto `Problema` —, o que prepara o
esqueleto para modalidades e anatomias futuras. Só a *tarefa* mudou; o protocolo e a fábrica de
arquiteturas permaneceram constantes.

## 12.1 · Síntese do que foi construído

A v3.1 herda **integralmente** o pipeline da v3.0 — ambiente, protocolo, dados, pré-processamento,
arquiteturas, laço de treino, inferência e avaliação — e altera **um único elemento**: a função de
perda. Especificamente:

- a **fábrica de perdas** `criar_perda()` e o seletor **`PERDA`** (Seção 7.1), espelhando
  `criar_modelo()` + `ARQUITETURA` da Seção 6. A perda deixa de ser uma constante escondida no
  código e passa a ser um **parâmetro declarado do experimento**;
- **`DiceFocalLoss`** como braço principal — mesmo termo Dice da v3.0, com a Cross-Entropy
  substituída por Focal (`γ = 2,0`), de modo que o gradiente se concentre nos voxels difíceis;
- **`dice_ce` preservada como controle**, para que a comparação seja **pareada** (mesma sessão,
  mesma `SEED`, mesmo cache) em vez de contra números de outra execução;
- **versionamento por experimento**: `resultados/figado_multiclasse/<arquitetura>__<perda>/`, com
  **guarda anti-sobrescrita** que aborta a execução se a assinatura `(arquitetura, perda, problema,
  versão)` divergir da que já estiver no disco — o §8.3 deixa de ser uma promessa e vira um
  mecanismo;
- **Seção 11 estendida**: lê os formatos da v3.0 e da v3.1 lado a lado e cruza `arquitetura × perda`
  no Dice de tumor.

> 🏛️ **O método, não o número.** A contribuição desta versão não é um Dice melhor — pode nem
> haver um. É ter transformado *"a perda talvez seja a culpada"* numa **hipótese testável, testada
> sob protocolo controlado, com o braço de controle rodando ao lado** e o resultado preservado
> qualquer que ele seja.

## 12.2 · Limitações assumidas

Coerente com o rigor científico do DPC (§10), reconhecemos explicitamente os limites desta versão:

- **A causa (ii) continua de pé** — esta é a limitação central da v3.1. O `spatial_size =
  (128, 128, 64)` redimensiona o volume inteiro e pode reduzir lesões pequenas a pouquíssimos
  voxels. **Nenhuma função de perda recupera informação destruída no pré-processamento.** Atacar
  isso (amostragem focada na lesão, `RandCropByPosNegLabeld` ou equivalente) é o eixo da **v3.2**,
  deliberadamente fora do escopo aqui — um eixo por vez.
- **Uma única arquitetura-âncora** — o experimento roda sobre a **U-Net 3D**. Um efeito da perda
  observado nela **não se generaliza automaticamente** às outras quatro; reconstruir a ilha com
  SegResNet, DynUNet, UNETR e SwinUNETR sob a perda vencedora é trabalho pendente, não concluído.
- **`γ = 2,0` não foi ajustado** — é o valor de referência da literatura, adotado sem busca. Uma
  varredura de `γ` (ou de `lambda_dice`/`lambda_focal`) seria **outro eixo**, e afiná-lo aqui
  contaminaria a comparação com a v3.0.
- **Sem pesos de classe explícitos** — a Focal pondera por **dificuldade**, não por **frequência**.
  Pesos por classe (`weight=`) seriam mais um grau de liberdade a justificar; ficam registrados
  como possibilidade, não usados.
- **`GeneralizedDiceLoss` disponível, mas não recomendada neste regime** — com `batch_size = 1` e
  volumes sem tumor no *ground truth*, seus pesos por classe degeneram.
- **`learning_rate = 1e-5` mantido** — herdado sem ajuste, para preservar a comparabilidade.
- **Computação** — o Colab impõe limites variáveis de GPU, memória e tempo (cache em disco,
  `num_workers=0`, um experimento por vez).
- **Ameaças à validade (§10.2)** — com **uma única semente**, uma diferença pequena entre perdas
  pode ser ruído de inicialização; para uma afirmação forte seriam necessárias repetições com
  sementes distintas. Dice/HD95 seguem sendo aproximações, combinadas com inspeção visual.

## 12.3 · O caminho adiante (roadmap)

A força da plataforma está em evoluir **sem perder comparabilidade**: dentro de uma ilha, *só um
eixo muda*. A v3.0 mudou a **tarefa**; a v3.1 muda a **perda**; a v3.2 mudará a **amostragem**. Cada
degrau isolado é o que permite dizer *por que* um número mudou.

| Versão | Eixo | Situação |
|---|---|---|
| **1.0–2.0** | Arquitetura (U-Net → CNNs → Transformers), fígado binário | ✅ concluídas |
| **3.0 — Multiclasse** | Tarefa: fígado + tumor (CT, MSD Task03) | ✅ concluída |
| **3.1 — Perda p/ desbalanceamento** | **Função de perda** (`DiceFocal` vs. `DiceCE`) | ✅ **esta versão** |
| **3.2 — Amostragem focada na lesão** | Pré-processamento: *crop* por classe positiva vs. *resize* global | ⏳ próxima |
| **3.3 — Ilha completa** *(condicional)* | Replicar a perda vencedora nas 5 arquiteturas | ⏳ prevista |
| **4.0 — MRI** | Modalidade: janela HU → z-score, multi-canal | ⏳ prevista |
| **5.0 — Novos órgãos / benchmarks** | Anatomia + múltiplas tasks do MSD | ⏳ prevista |
| **6.0–7.0 — Cérebro** | Convergência: MRI multi-sequência + multiclasse (Task04 → Task01) | 🎯 objetivo |

> 🧭 **A v3.2 depende do que a v3.1 responder.** Se `dice_focal` afastar o tumor de zero, a v3.2
> parte de uma base viva e mede um **ganho incremental**. Se não afastar, a v3.1 terá **eliminado
> uma hipótese** — e a v3.2 passa a ser a candidata principal, com evidência por trás. Nos dois
> casos o projeto avança: é assim que um eixo por vez paga.

Perspectivas de mais longo prazo (§10.4): adaptação de domínio e experimentos multicêntricos,
aprendizado semi/auto-supervisionado, federado, modelos fundacionais e interpretabilidade
(Grad-CAM/SHAP) — trilhas que se somam **depois** de o esqueleto cobrir as tarefas-alvo.

## 12.4 · Considerações éticas

Reforçando o **Aviso** inicial (§10.3): esta plataforma usa exclusivamente dados **públicos e
anonimizados** e destina-se **apenas a pesquisa, ensino e desenvolvimento**. **Não é um dispositivo
médico** e não deve orientar decisão clínica sem validação específica, aprovação ética e
conformidade regulatória. A responsabilidade final permanece sempre com o profissional habilitado.

## 12.5 · Manifesto científico

O legado deste projeto não está em uma arquitetura ou tarefa específica, mas na **infraestrutura
metodológica** capaz de acompanhar a evolução da IA sem perder consistência científica (§11.3 do DPC):

- a **ciência precede a tecnologia** — arquiteturas e tarefas evoluem, os princípios permanecem;
- **reprodutibilidade** é indispensável e **transparência** é obrigatória;
- comparações só são justas sob **exatamente o mesmo protocolo**, dentro da **mesma ilha**;
- a **qualidade dos dados** tem prioridade sobre a sofisticação do modelo;
- a **medicina orienta** a IA, e a colaboração multidisciplinar é indispensável.

---

> *"Tecnologias mudam. Frameworks evoluem. Modelos são substituídos. O rigor científico permanece."*
> — Manifesto Científico da Plataforma (DPC §11.4)

---

### 🎯 Fim do pipeline v3.0

Da configuração do ambiente à comparação entre arquiteturas, o notebook agora demonstra a
plataforma **abrindo um novo eixo de problema** — a segmentação multiclasse de fígado e tumor —
sem abandonar a comparabilidade nem a reprodutibilidade, e já parametrizada para os próximos
passos rumo a novas modalidades e anatomias.